### Benchmark de *Multiple Instance Learning* e *Foundation Models* em fragmentômica de *cfDNA*

---

## 1. A pergunta

Quando uma pessoa faz um exame de biópsia líquida, o plasma contém milhões de fragmentos de DNA livre (*cfDNA*). Em câncer inicial, menos de 1% deles vem do tumor. Um modelo de linguagem genômica (*Foundation Model*) pode ler cada fragmento e produzir um vetor de 256 números. Mas entre esses vetores e um diagnóstico existe uma cadeia de escolhas de engenharia: como combinar os fragmentos, onde aplicar não linearidades, que classificador usar.

Este caderno faz duas perguntas:

- **Onde o sinal se perde?** Construímos uma escada de seis degraus ligando o classificador mais simples ao mais complexo, mudando **um elemento por vez**. Isso atribui a cada decisão de engenharia a sua fatia exata.

- **O encoder semântico pode agregar sinal?** Tentamos reconstruir o vetor do *Foundation Model* a partir de duas coisas simples: a composição de bases dos fragmentos e o lugar onde caíram no genoma. O que sobra é o candidato ao que o pré-treino acrescentou.

---

## 2. O regime de sinal fraco

Um fragmento isolado separa muito pouco os grupos, a diferença é da ordem de centésimos de desvio padrão. Só a média sobre milhares torna o sinal visível.

Nesse regime, uma não linearidade aplicada **antes** da média opera em boa parte sobre variação não informativa. Uma ReLU zera valores negativos: sobre ruído aproximadamente simétrico, ela corta metade da distribuição para um lado só e altera a média antes que o cancelamento aconteça. Isso não vale para qualquer função nem em qualquer regime, é uma afirmação sobre **estes dados**, e o degrau ORDEM a mede.

---

## 3. Desenho amostral

| Partição              | Coorte                             | n   | Papel                                                 |
| :-------------------- | :--------------------------------- | --: | :---------------------------------------------------- |
| **Desenvolvimento**   | Baltimore (Cristiano et al., 2019) | 537 | Validação cruzada de 5 dobras; toda a análise forense |
| **Avaliação externa** | Hong Kong (Jiang et al., 2015)     | 225 | Inferência apenas; estatuto exploratório              |

O treino interno é um detector **pan-câncer**: 276 casos de oito tipos tumorais contra 261 controles saudáveis. **Baltimore não contém nenhum caso de carcinoma hepatocelular.**

A avaliação externa testa **carcinoma hepatocelular** (90 casos) contra saudáveis (32). Hepatite B (67) e cirrose (36) não entram na AUROC; aparecem na Figura 3 para mostrar o que o modelo faz diante de inflamação sem tumor.

Quando o desempenho cai na coorte externa, **três causas se confundem**: tipo tumoral não visto no treino, ascendência (ocidental / Asiática) e protocolo de sequenciamento. Não é validação externa da mesma tarefa clínica, e sim **generalização para uma coorte não homóloga**. O resultado primário é **interno**.

As Células 1 a 3 usam exclusivamente Baltimore. Hong Kong só aparece na Célula 4.

---

## 4. As representações comparadas

Cada paciente é descrito por 5.000 fragmentos. O que muda é o que medimos sobre cada um:

| Representação              |   Dim. | O que é, em palavras simples                                               | Resolução |
| :------------------------- | -----: | :------------------------------------------------------------------------- | :-------- |
| **Caduceus**               |    256 | O que o *Foundation Model* «vê» na sequência ao redor do fragmento         | Base      |
| **4-mers**                 |    256 | Com que frequência cada combinação de 4 letras aparece                     | Base      |
| **Comprimento**            |    1   | O tamanho do fragmento, em pares de base                                   | Molécula  |
| **Cobertura DELFI**        |  1.614 | Quantos fragmentos curtos e longos há em cada janela de 5 milhões de bases | Megabase  |
| **Ocupação de promotores** | 39.614 | Em qual promotor cada fragmento caiu                                       | Promotor  |

A ocupação é 88% zeros — com 5.000 fragmentos e 39.614 alvos, a maioria fica vazia. Ela entra na reconstrução do encoder e no teste de coorte, não como classificador.

---

## 5. Cinco repetições, e duas formas de incerteza

Redes treinadas sobre bolsas de 5.000 instâncias com sinal fraco têm objetivos de alta variância: a mesma arquitetura, com outra semente, pode chegar a outro resultado. Todo o pipeline é executado **cinco vezes**, com partições e inicializações independentes.

Cada degrau é reportado por **dois estimadores diferentes**, cada um com a sua incerteza:

| Estimador                          | Incerteza             | O que varia                | O que fica fixo      |
| :--------------------------------- | :-------------------- | :------------------------- | :------------------- |
| Média das diferenças **por dobra** | desvio entre sementes | partições e inicializações | os pacientes         |
| Diferença do **escore agrupado**   | bootstrap pareado     | a amostra de pacientes     | os modelos treinados |

Eles não medem a mesma quantidade. A escada soma o primeiro; o segundo é um teste independente sobre os mesmos pacientes.

---


## 6. Procedência e Preparação do Conjunto de Dados

Os dados de entrada provêm de sequenciamentos genômicos totais (*WGS*) de baixa cobertura padronizados no repositório FinaleDB (Zheng et al., 2020), cobrindo as coortes de Baltimore (Cristiano et al., 2019; $n=537$) e Hong Kong (Jiang et al., 2015; $n=225$):

1. **Seleção e Amostragem de Fragmentos:** A partir dos alinhamentos contra o genoma de referência ($\text{hg38}$), selecionaram-se moléculas de *cfDNA* cujas coordenadas interceptam um painel curado de 39.614 regiões promotoras gênicas humanas (sítios de início de transcrição e cromatina aberta). Para paridade metodológica estrita, cada paciente foi representado por uma bolsa fixa de exatamente 5.000 fragmentos genômicos reais (sem *padding* artificial).
2. **Extração de Sequência e *Embeddings* do Caduceus (256d):** Para cada fragmento capturado, extraiu-se a sequência nucleotídica da referência $\text{hg38}$ ao longo de sua extensão acrescida de uma margem simétrica de $\pm 6\text{ pb}$ além dos sítios de clivagem (janela mediana total de 179 pb). As sequências foram processadas pelo codificador genômico bidirecional congelado *Caduceus-Ph* (`d_model = 256`), gerando um vetor latente denso por instância.
3. **Composição de 4-mers (256d):** Calculou-se a frequência relativa empírica de todos os 256 possíveis tetranucleotídeos (oligonucleotídeos de 4 bases) ao longo de toda a janela lida de cada fragmento.
4. **Comprimento Físico (1d) e Identificador de Promotor:** Extraiu-se o comprimento da molécula em pares de bases a partir dos pontos de quebra 5′ e 3′, e anotou-se o índice categórico do promotor alvo atingido.
5. **Cobertura Macroscópica DELFI 5 Mb (1.614d):** Calculada no nível do paciente a partir do sequenciamento genômico total (*WGS* completo), compreendendo a cobertura padronizada e as razões entre fragmentos curtos (100–150 pb) e longos (151–220 pb) distribuídas ao longo de janelas autossômicas de 5 megabases.

---


## 6. Quality Gates

A execução é interrompida se qualquer um falhar:

| Portão                           | O que protege                                                                                             |
| :------------------------------- | :-------------------------------------------------------------------------------------------------------- |
| **Consistência estrutural**      | Formas e tipos de todos os arquivos                                                                       |
| **Finitude**                     | Completa em DELFI e comprimentos; triagem amostral nos embeddings                                         |
| **Sem preenchimento artificial** | Bolsas incompletas corromperiam médias e pesos de atenção                                                 |
| **Vetor idêntico**               | O tensor consumido pela regressão e pela rede deve ser o mesmo                                            |
| **Comutação linear/média**       | A camada afim comuta com a média; logo as duas arquiteturas do degrau ORDEM diferem só na posição da ReLU |
| **Escala do controle nulo**      | O canal de comprimento tem de estar padronizado em todos os braços                                        |
| **Identidade telescópica**       | A soma dos degraus deve igualar a diferença entre extremos                                                |
| **Substrato do resíduo**         | Se a reconstrução linear falhar, a análise do resíduo é abortada                                          |

---



In [1]:
# =============================================================================
# ===== CÉLULA 1 — SETUP, DADOS E REGIME DE SINAL =============================
# =============================================================================

import json, time, hashlib, warnings, platform
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.linear_model import LogisticRegression, RidgeCV
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, StratifiedShuffleSplit
from sklearn.metrics import roc_auc_score, r2_score
from sklearn.exceptions import ConvergenceWarning
from scipy.stats import spearmanr, beta

# Avisos de convergência não são silenciados: um modelo que não convergiu é
# informação, não ruído. São contados e relatados.
warnings.simplefilter("always", ConvergenceWarning)
T_INICIO = time.time()

# -----------------------------------------------------------------------------
# Parâmetros
# -----------------------------------------------------------------------------
CFG = dict(
    SEMENTES             = (1807, 2024, 3141, 5926, 8979),
    N_FRAGMENTOS         = 5000,
    N_DOBRAS             = 5,
    FRACAO_PARADA        = 0.15,
    # Treinamento
    MAX_EPOCAS           = 80,
    MIN_EPOCAS           = 12,
    PACIENCIA            = 10,
    JANELA_SUAVIZACAO    = 5,
    TAXA_APRENDIZADO     = 3e-4,
    DECAIMENTO_PESO      = 1e-4,
    TAMANHO_LOTE         = 16,
    DIM_OCULTA           = 128,
    CLIP_GRADIENTE       = 5.0,
    # Triagem de desempenho mínimo, usada só na análise de sensibilidade.
    # Aplicada ao máximo da curva SUAVIZADA no conjunto de parada: o máximo
    # da curva bruta cruza qualquer limiar fixo com frequência apreciável,
    # porque são dezenas de avaliações ruidosas.
    LIMIAR_PARADA        = 0.60,
    # Pré-processamento
    PADRONIZA_BOLSA      = True,
    BOLSAS_NA_GPU        = True,
    N_COMPONENTES_OCUP   = 64,
    COMPRIMENTO_MIN_HIST = 60,
    COMPRIMENTO_MAX_HIST = 420,
    FRACAO_MAX_FORA_HIST = 0.01,
    # Estatística
    N_PERMUTACOES        = 200,
    N_BOOTSTRAP          = 2000,
    N_BOOTSTRAP_VARIANCIA= 400,
    K_COMPONENTES_PCA    = (1, 2, 5, 10, 25),
    ESPECIFICIDADE_ALVO  = 0.95,
    R2_ABORTA_RESIDUO    = 0.05,
    R2_INTERPRETA_RESIDUO= 0.30,
    # Diagnóstico
    N_PACIENTES_FORENSE  = 60,
    N_PACIENTES_FINITUDE = 30,
    N_PACIENTES_POR_CLASSE_D = 60,
    JANELA_EXTRA_ENCODER = 12.0,
    # Duas verificações de robustez que não produzem nenhum número citado no
    # artigo: a distribuição nula do vetor completo do Caduceus e a versão
    # padronizada da PCA da ocupação. Ligue para auditar; custam ~6 minutos.
    VERIFICACOES_EXTRA   = False,
)
SEMENTE_MESTRA = CFG["SEMENTES"][0]

HASH_CFG = hashlib.sha256(
    json.dumps(CFG, sort_keys=True, default=str).encode()).hexdigest()[:16]

np.random.seed(SEMENTE_MESTRA)
torch.manual_seed(SEMENTE_MESTRA)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DISPOSITIVO = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ESTUDO_DEV, ESTUDO_EXT = "cristiano2019_delfi_wgs", "jiang2015_hcc_wgs"
SAUDAVEL, HCC = {"healthy"}, "liver_cancer"

print("=" * 96)
print("ONDE O SINAL SE PERDE ENTRE 5.000 FRAGMENTOS E UM DIAGNÓSTICO")
print("Auditoria de Multiple Instance Learning em fragmentômica de cfDNA")
print("=" * 96)
print(f"Assinatura da configuração : {HASH_CFG}")
print(f"Repetições independentes   : {len(CFG['SEMENTES'])} sementes")
print(f"Coorte externa             : exploratória, tarefa distinta")
print(f"Verificações extras        : {'ligadas' if CFG['VERIFICACOES_EXTRA'] else 'desligadas'}")
print(f"Hardware                   : {DISPOSITIVO}")

# -----------------------------------------------------------------------------
# Glossário dos braços. Nomes por extenso em toda a saída; as formas curtas
# só aparecem em eixos de gráfico, onde não cabe o nome inteiro.
# -----------------------------------------------------------------------------
NOME = {
    "cad_lin":          "Caduceus (Média → Logística)",
    "cad_vec_lin":      "Caduceus + Comprimento (Média → Logística)",
    "cad_vec_lin_frag": "Caduceus + Comprimento (Média → Logística, escala de fragmento)",
    "cad_vec_mlpdir":   "Caduceus + Comprimento (Média → MLP)",
    "cad_vec_mlp":      "Caduceus + Comprimento (Média → Projeção → MLP)",
    "cad_mean":         "Caduceus + Comprimento (Projeção → Média → MLP)",
    "cad_att":          "Caduceus + Comprimento (Projeção → Atenção MIL)",
    "kmer_vec_mlp":     "4-mers + Comprimento (Média → Projeção → MLP)",
    "kmer_mean":        "4-mers + Comprimento (Projeção → Média → MLP)",
    "kmer_att":         "4-mers + Comprimento (Projeção → Atenção MIL)",
    "length":           "Somente Comprimento (Atenção MIL) [controle nulo]",
    "delfi":            "Cobertura DELFI 5 Mb (MLP)",
    "rep_kmer":         "Composição de 4-mers (Média → Logística)",
    "rep_len":          "Histograma de Comprimentos (Logística)",
    "rep_delfi":        "Cobertura DELFI 5 Mb (Logística)",
    "rep_cad_resid":    "Caduceus — Resíduo após 4-mers e Ocupação (Logística)",
}
CURTO = {
    "cad_lin": "Cad: Média→Log",              "cad_vec_lin": "Cad+Comp: Média→Log",
    "cad_vec_lin_frag": "Cad+Comp: Média→Log (frag)",
    "cad_vec_mlpdir": "Cad+Comp: Média→MLP",  "cad_vec_mlp": "Cad+Comp: Média→Proj→MLP",
    "cad_mean": "Cad+Comp: Proj→Média→MLP",   "cad_att": "Cad+Comp: Proj→Atenção",
    "kmer_vec_mlp": "4mer: Média→Proj→MLP",   "kmer_mean": "4mer: Proj→Média→MLP",
    "kmer_att": "4mer: Proj→Atenção",         "length": "Só Comprimento: Atenção",
    "delfi": "DELFI: MLP",                    "rep_kmer": "4mers: Log",
    "rep_len": "Comprimento: Log",            "rep_delfi": "DELFI: Log",
    "rep_cad_resid": "Cad Resíduo: Log",
}
def nome(k): return NOME.get(k, k)

print("\n" + "-" * 96)
print("AMBIENTE DE SOFTWARE")
print("-" * 96)
for lib, ver in [("python", platform.python_version()), ("numpy", np.__version__),
                 ("pandas", pd.__version__), ("scikit-learn", sklearn.__version__),
                 ("torch", torch.__version__), ("seaborn", sns.__version__)]:
    print(f"  {lib:<16s} {ver}")
if torch.cuda.is_available():
    print(f"  {'gpu':<16s} {torch.cuda.get_device_name(0)}")

# -----------------------------------------------------------------------------
# HASH dos arquivos de entrada
# -----------------------------------------------------------------------------
PASTA = None
for c in Path("/kaggle/input").rglob("02_features"):
    if c.is_dir():
        PASTA = c; break
if PASTA is None:
    raise RuntimeError("Pasta '02_features' não encontrada. Anexe o dataset ao notebook.")
SAIDA = Path("/kaggle/working/04_results"); SAIDA.mkdir(parents=True, exist_ok=True)

ARQUIVOS = {"metadados": "metadata.csv", "caduceus": "caduceus_embeddings.npy",
            "delfi": "delfi_all.npy",    "kmers": "kmer_features.npy",
            "comprimento": "promoter_fragment_lengths.npy",
            "promotores": "promoter_ids.npy"}
for _, arq in ARQUIVOS.items():
    if not (PASTA / arq).exists():
        raise RuntimeError(f"Arquivo obrigatório ausente: {arq}")

print("\n" + "-" * 96)
print("ASSINATURA DOS ARQUIVOS DE ENTRADA (SHA-256)")
print("-" * 96)
print(f"  Entrada : {PASTA}")
print(f"  Saída   : {SAIDA}\n")
t0 = time.time(); hashes = {}
for _, arq in ARQUIVOS.items():
    h = hashlib.sha256()
    with open(PASTA / arq, "rb") as f:
        for bloco in iter(lambda: f.read(8 * 1024 * 1024), b""):
            h.update(bloco)
    hashes[arq] = h.hexdigest()
    print(f"  {arq:<34s} {h.hexdigest()[:24]}...")
HASH_DADOS = hashlib.sha256(json.dumps(hashes, sort_keys=True).encode()).hexdigest()[:16]
print(f"\n  Assinatura agregada : {HASH_DADOS}   ({time.time()-t0:.0f} s)")

# mmap_mode="r" mapeia o arquivo sem carregá-lo na RAM. As matrizes de embedding
# têm ~2 GB cada; lê-se só o pedaço necessário.
META  = pd.read_csv(PASTA / ARQUIVOS["metadados"])
CAD   = np.load(PASTA / ARQUIVOS["caduceus"], mmap_mode="r")
KMER  = np.load(PASTA / ARQUIVOS["kmers"],    mmap_mode="r")
COMP  = np.load(PASTA / ARQUIVOS["comprimento"])
PROM  = np.load(PASTA / ARQUIVOS["promotores"], mmap_mode="r")
DELFI = np.load(PASTA / ARQUIVOS["delfi"]).astype(np.float32)

N_PAC = len(META)
D_EMB = CAD.shape[2]

# -----------------------------------------------------------------------------
# GATE — consistência estrutural e finitude.
#
# Verifica que todos os arquivos têm o mesmo número de linhas, a mesma
# geometria, e que nenhum valor é infinito ou indefinido. 
# -----------------------------------------------------------------------------
print("\n" + "=" * 96)
print("GATE — CONSISTÊNCIA ESTRUTURAL E FINITUDE")
print("=" * 96)
falhas = []
for nm, arr in [("Caduceus", CAD), ("4-mers", KMER), ("Comprimentos", COMP),
                ("Promotores", PROM), ("DELFI", DELFI)]:
    if arr.shape[0] != N_PAC:
        falhas.append(f"{nm}: {arr.shape[0]} linhas, metadados têm {N_PAC}")
for nm, arr in [("Caduceus", CAD), ("4-mers", KMER), ("Comprimentos", COMP),
                ("Promotores", PROM)]:
    if arr.shape[1] != CFG["N_FRAGMENTOS"]:
        falhas.append(f"{nm}: {arr.shape[1]} fragmentos por paciente, "
                      f"esperado {CFG['N_FRAGMENTOS']}")
if KMER.shape[2] != D_EMB:
    falhas.append(f"4-mers têm {KMER.shape[2]} dimensões, Caduceus tem {D_EMB}")
if not np.isfinite(DELFI).all():
    falhas.append("DELFI contém valores não finitos")
if not np.isfinite(COMP).all():
    falhas.append("Comprimentos contêm valores não finitos")
# Nos embeddings a verificação é uma TRIAGEM amostral: varrer 4 GB duas vezes
# não muda nenhum número, e corrupção sistemática aparece na amostra.
_am_fin = np.sort(np.random.RandomState(SEMENTE_MESTRA).choice(
    N_PAC, min(CFG["N_PACIENTES_FINITUDE"], N_PAC), replace=False))
for nm, arr in [("Caduceus", CAD), ("4-mers", KMER)]:
    if not np.isfinite(np.asarray(arr[_am_fin], dtype=np.float32)).all():
        falhas.append(f"{nm} contém valores não finitos (triagem de {len(_am_fin)} pacientes)")
if PROM.max() < 0 or PROM.min() < -1:
    falhas.append("Identificadores de promotor fora da faixa esperada")
if falhas:
    for f in falhas: print(f"  [FALHA] {f}")
    raise RuntimeError("Estrutura dos arquivos inconsistente.")

N_PROM = int(PROM.max()) + 1   # definido só depois de PROM passar no gate de faixa

print("  Aprovado: formas e tipos conferem.")
print(f"  Finitude completa  : DELFI, comprimentos")
print(f"  Finitude amostral  : Caduceus, 4-mers ({len(_am_fin)} pacientes sorteados)")

# -----------------------------------------------------------------------------
# GATE — as bolsas estão completas?
#
# Se um paciente tivesse menos de 5.000 fragmentos e o resto fosse preenchido
# com zeros, a padronização transformaria esses zeros em valores não nulos. A
# média e os pesos da atenção passariam a depender da quantidade de
# preenchimento, não da biologia.
# -----------------------------------------------------------------------------
n_zero = int((COMP == 0).sum())
print("\n" + "-" * 96)
print("GATE — BOLSAS COMPLETAS (AUSÊNCIA DE PREENCHIMENTO ARTIFICIAL)")
print("-" * 96)
print(f"  Posições com comprimento zero : {n_zero} de {COMP.size}")
if n_zero > 0:
    raise RuntimeError(
        f"{n_zero} posições vazias detectadas. Sem máscara explícita, médias e "
        f"pesos de atenção ficariam contaminados pela fração de preenchimento.")
print(f"  Aprovado: todos os {N_PAC} pacientes têm {CFG['N_FRAGMENTOS']} fragmentos reais.")

print("\n" + "-" * 96)
print("COMPOSIÇÃO DAS COORTES")
print("-" * 96)
print(META.groupby(["estudo", "condicao"]).size().to_string())

IDX_DEV = np.where((META["estudo"] == ESTUDO_DEV).values)[0]
IDX_EXT = np.where((META["estudo"] == ESTUDO_EXT).values)[0]

# -----------------------------------------------------------------------------
# ESCALA DOS CANAIS E QUANTO A BOLSA REALMENTE CARREGA
#
#
# (a) ESCALA. O comprimento de um fragmento varia de 60 a 420 pares de base.
#     Cada dimensão do Caduceus varia em torno de 0,004. Sem padronização, o
#     comprimento dominaria a rede por duas ordens de grandeza.
#
# (b) PONTO DE CROSSOVER DA VARIÂNCIA. Cada dimensão varia por dois motivos:
#     fragmentos do mesmo paciente diferem entre si (variância DENTRO, σ²d) e
#     pacientes diferem entre si (variância ENTRE, σ²e). A média de n
#     fragmentos tem variância
#
#          σ²e  +  σ²d / n
#
#     O segundo termo encolhe com n; o primeiro não. Onde os dois se igualam,
#     n* = σ²d/σ²e, o erro de estimar a média deixa de ser o termo dominante.
#     A σ²e observada já contém σ²d/5000, então a corrigimos. O cálculo não usa
#     rótulo de doença.
# -----------------------------------------------------------------------------
print("\n" + "=" * 96)
print("ESCALA DOS CANAIS E PONTO DE CROSSOVER DA VARIÂNCIA (só Baltimore)")
print("=" * 96)
amo = np.sort(np.random.RandomState(SEMENTE_MESTRA).choice(
    IDX_DEV, size=min(CFG["N_PACIENTES_FORENSE"], len(IDX_DEV)), replace=False))

def crossover_variancia(bolsa, n_boot, semente):
    """n* = σ²dentro / σ²entre, com intervalo por reamostragem de PACIENTES."""
    var_d_pac = bolsa.var(axis=1)
    medias    = bolsa.mean(axis=1)
    def calcula(idx):
        vd = var_d_pac[idx].mean(axis=0)
        vo = medias[idx].var(axis=0, ddof=1)
        ve = np.maximum(vo - vd / CFG["N_FRAGMENTOS"], 1e-24)
        return vd / ve
    n_est = calcula(np.arange(len(bolsa)))
    rng = np.random.RandomState(semente); n = len(bolsa)
    boots = [float(np.median(calcula(rng.randint(0, n, n)))) for _ in range(n_boot)]
    return (float(np.median(n_est)),
            float(np.percentile(n_est, 25)), float(np.percentile(n_est, 75)),
            float(np.percentile(boots, 2.5)), float(np.percentile(boots, 97.5)))

forense = {}
for nm, arr in [("Caduceus", CAD), ("4-mers", KMER)]:
    b = np.asarray(arr[amo], dtype=np.float32)
    ne, q25, q75, blo, bhi = crossover_variancia(
        b, CFG["N_BOOTSTRAP_VARIANCIA"], SEMENTE_MESTRA)
    forense[nm] = dict(dp_frag=float(b.std(axis=1).mean()),
                       dp_medias=float(b.mean(axis=1).std(axis=0, ddof=1).mean()),
                       n_estrela=ne, iqr_dim=(q25, q75), ic_pacientes=(blo, bhi))
    f = forense[nm]
    print(f"\n  {nm}")
    print(f"    Desvio entre fragmentos do mesmo paciente (σd) : {f['dp_frag']:.5f}")
    print(f"    Desvio entre pacientes, observado (σe)         : {f['dp_medias']:.5f}")
    print(f"    Crossover n* = σ²d/σ²e (mediana das dimensões) : {ne:.0f}")
    print(f"      dispersão entre dimensões (quartis)          : {q25:.0f} – {q75:.0f}")
    print(f"      IC 95% por reamostragem de pacientes         : {blo:.0f} – {bhi:.0f}")
    del b

N_ESTRELA = forense["Caduceus"]["n_estrela"]
N_EST_LO, N_EST_HI = forense["Caduceus"]["ic_pacientes"]

print(f"\n  LEITURA: até cerca de {N_ESTRELA:.0f} fragmentos o erro de estimação da média")
print(f"  domina; a partir daí, domina a variação entre pacientes. Fragmentos de um")
print(f"  mesmo paciente compartilham região genômica, protocolo e composição, não")
print(f"  contam como observações independentes. É o efeito de desenho clássico de")
print(f"  amostragem por conglomerado.")

print(f"\n  Escala relativa na primeira camada:")
print(f"    Comprimento, depois de padronizado : desvio 1,000")
for nm in ["Caduceus", "4-mers"]:
    dp = forense[nm]["dp_frag"]
    print(f"    Cada dimensão de {nm:<10s}         : desvio {dp:.5f}  → {1.0/dp:.0f}x menor")
print(f"  Por isso a padronização de cada canal é obrigatória, e está ativa.")

# -----------------------------------------------------------------------------
# O que estes «4-mers» medem?
#
# Se cada fragmento contribuísse com UM 4-mer (o motivo de corte), cada dimensão
# seria um ensaio de Bernoulli com p = 1/256. Se contamos os 4-mers ao longo de
# toda a janela lida pelo encoder, a média de muitos ensaios reduz o desvio.
# Comparar as duas previsões com o observado diz qual das duas descreve o
# arquivo. Importa porque o achado central é «o encoder é composição de bases» —
# e o leitor precisa saber de que composição estamos falando.
# -----------------------------------------------------------------------------
comp_amo   = COMP[amo]
comp_med   = float(np.median(comp_amo[comp_amo > 0]))
janela     = comp_med + CFG["JANELA_EXTRA_ENCODER"]
dp_motivo  = float(np.sqrt((1/256.0) * (1 - 1/256.0)))
dp_composi = dp_motivo / np.sqrt(max(janela - 3.0, 1.0))
dp_obs     = forense["4-mers"]["dp_frag"]

print("\n" + "-" * 96)
print("O QUE OS 4-MERS MEDEM")
print("-" * 96)
print(f"  Janela lida pelo encoder : {janela:.0f} pb (fragmento mediano {comp_med:.0f} pb)")
print(f"  Desvio por dimensão      : motivo de corte {dp_motivo:.4f} · "
      f"composição da janela {dp_composi:.4f} · OBSERVADO {dp_obs:.5f}")
print(f"  O observado é {dp_obs/dp_composi:.2f}x a previsão de composição de janela e "
      f"{dp_obs/dp_motivo:.3f}x a de motivo.")
print(f"  Compatível, em ordem de grandeza, com composição da janela. As 4 bases do")
print(f"  ponto de corte pesam {100*4.0/janela:.1f}% da janela: o motivo terminal não é medido aqui.")

# -----------------------------------------------------------------------------
print("\n" + "-" * 96)
print("LIMITES DO QUE ESTA CÉLULA MEDE")
print("-" * 96)
print("  • Os arquivos de features não carregam identificador de paciente. Que a")
print("    linha 173 de cada arquivo seja o mesmo paciente é pressuposto da etapa")
print("    de geração; aqui só a dimensão é verificada.")
print("  • A finitude dos embeddings é uma triagem em "
      f"{len(_am_fin)} pacientes, não uma varredura.")
print("  • n* é um ponto de crossover sob o modelo de variância adotado, que supõe")
print("    correlação intra-bolsa homogênea. Não é tamanho amostral efetivo no")
print("    sentido clássico, e não prova que os demais fragmentos sejam inúteis.")
print("  • A amostra mistura casos e controles: a doença contribui para σ²e e")
print("    portanto reduz n*. O valor é conservador.")
print("  • A semântica dos 4-mers é INFERIDA por ordem de grandeza, não documentada")
print("    aqui. Os 4-mers sobrepostos não são independentes, então a previsão de")
print("    composição é otimista. A receita de geração pertence aos Methods.")

print("\n" + "=" * 96); print("FIM DA CÉLULA 1"); print("=" * 96)

# =============================================================================
# ===== FIM CÉLULA 1 ==========================================================
# =============================================================================

ONDE O SINAL SE PERDE ENTRE 5.000 FRAGMENTOS E UM DIAGNÓSTICO
Auditoria de Multiple Instance Learning em fragmentômica de cfDNA
Assinatura da configuração : 404e0e2f7ea254e9
Repetições independentes   : 5 sementes
Coorte externa             : exploratória, tarefa distinta
Verificações extras        : desligadas
Hardware                   : cuda

------------------------------------------------------------------------------------------------
AMBIENTE DE SOFTWARE
------------------------------------------------------------------------------------------------
  python           3.12.13
  numpy            2.0.2
  pandas           2.3.3
  scikit-learn     1.6.1
  torch            2.10.0+cu128
  seaborn          0.13.2
  gpu              Tesla T4

------------------------------------------------------------------------------------------------
ASSINATURA DOS ARQUIVOS DE ENTRADA (SHA-256)
------------------------------------------------------------------------------------------------
  Entrada

In [2]:
# =============================================================================
# ===== CÉLULA 2 — REPRESENTAÇÕES E CONTEÚDO DO ENCODER =======================
# =============================================================================

print("=" * 96)
print("CÉLULA 2 — REPRESENTAÇÕES E CONTEÚDO DO ENCODER")
print("=" * 96)

MAPA = {"healthy": 0, "cirrhosis": 0, "hepatitis_b": 0,
        "bile_duct_cancer": 1, "breast_cancer": 1, "colorectal_cancer": 1,
        "duodenal_cancer": 1, "gastric_cancer": 1, "lung_cancer": 1,
        "ovarian_cancer": 1, "pancreatic_cancer": 1, "liver_cancer": 1}
sem_mapa = sorted(set(META["condicao"].dropna()) - set(MAPA))
if sem_mapa:
    raise RuntimeError(f"Condições clínicas sem rótulo definido: {sem_mapa}")
rotulos = META["condicao"].map(MAPA).astype(np.int8).to_numpy()

Y_DEV = rotulos[IDX_DEV].astype(int)
N_DEV = len(IDX_DEV)
hcc_dev = int((META.loc[IDX_DEV, "condicao"] == HCC).sum())

print(f"\n  Desenvolvimento : {N_DEV} pacientes "
      f"({Y_DEV.sum()} casos / {(1-Y_DEV).sum()} controles)")
print(f"  Externa         : {len(IDX_EXT)} pacientes")
print(f"  Carcinoma hepatocelular no treino : {hcc_dev}")
if hcc_dev == 0:
    print("  As duas tarefas diferem: treino pan-câncer, avaliação num tipo tumoral")
    print("  ausente do treino, noutra população e noutro protocolo.")

# -----------------------------------------------------------------------------
# Vetores por paciente.
#
# A leitura em blocos ordena os índices e
# depois desfaz a ordenação, para que cada linha continue correspondendo ao seu
# paciente. As linhas 0..N_DEV-1 são Baltimore; de N_DEV em diante, Hong Kong.
# -----------------------------------------------------------------------------
print("\n  Construindo vetores por paciente...")
t0 = time.time()
ordem    = np.concatenate([IDX_DEV, IDX_EXT])
ord_sort = np.sort(ordem)
inversa  = np.argsort(np.argsort(ordem))

POS = np.full(N_PAC, -1, dtype=np.int64)
POS[ordem] = np.arange(len(ordem))
if not (POS[ordem] >= 0).all() or not np.array_equal(ord_sort[inversa], ordem):
    raise RuntimeError("Falha de alinhamento entre índices de paciente e linhas.")
if not np.array_equal(ordem[:N_DEV], IDX_DEV) or not np.array_equal(ordem[N_DEV:], IDX_EXT):
    raise RuntimeError("As linhas a partir de N_DEV não correspondem à coorte externa.")

def media_por_paciente(matriz, lote=40):
    buf = np.zeros((len(ord_sort), matriz.shape[2]), dtype=np.float32)
    for i in range(0, len(ord_sort), lote):
        idx = ord_sort[i:i+lote]
        buf[i:i+len(idx)] = np.asarray(matriz[idx], dtype=np.float32).mean(axis=1)
    return buf[inversa]

V_CAD  = media_por_paciente(CAD)
V_KMER = media_por_paciente(KMER)
COMP_ORD = COMP[ord_sort].astype(np.float32)[inversa]
V_COMP   = COMP_ORD.mean(axis=1, keepdims=True)

BINS = np.arange(CFG["COMPRIMENTO_MIN_HIST"], CFG["COMPRIMENTO_MAX_HIST"] + 1, 3)
validos   = COMP_ORD[COMP_ORD > 0]
frac_fora = float(np.mean((validos < BINS[0]) | (validos > BINS[-1])))
V_HIST = np.stack([np.histogram(l[l > 0], bins=BINS, density=True)[0]
                   for l in COMP_ORD]).astype(np.float32)

prom_ord = np.asarray(PROM[ord_sort], dtype=np.int64)[inversa]
V_OCUP = np.zeros((len(ordem), N_PROM), dtype=np.float32)
for i, linha in enumerate(prom_ord):
    np.add.at(V_OCUP[i], linha[linha >= 0], 1.0)
V_OCUP /= np.maximum(V_OCUP.sum(axis=1, keepdims=True), 1.0)
del prom_ord
V_DELFI = DELFI[ordem]
frac_zeros = float((V_OCUP == 0).mean())
print(f"  concluído em {time.time()-t0:.0f} s")

if frac_fora > CFG["FRACAO_MAX_FORA_HIST"]:
    raise RuntimeError("Descarte excessivo de fragmentos fora do histograma.")
print(f"  Fragmentos fora do histograma de comprimento : {100*frac_fora:.4f}%")
print(f"  Zeros na matriz de ocupação de promotores    : {100*frac_zeros:.1f}%")

# -----------------------------------------------------------------------------
# Partições, definidas uma vez e compartilhadas por tudo: reconstrução do
# resíduo, permutações e treinamento.
#
# O conjunto de parada é estratificado. Ele é consultado a cada época para
# escolher o melhor ponto do treino; uma divisão aleatória poderia entregar
# 25 casos contra 39 controles em vez de 32/32, acrescentando ruído justamente
# à quantidade usada para seleção.
# -----------------------------------------------------------------------------
def constroi_dobras(semente, y=None):
    y = Y_DEV if y is None else y
    skf = StratifiedKFold(n_splits=CFG["N_DOBRAS"], shuffle=True, random_state=semente)
    dobras = []
    for itr, iva in skf.split(np.zeros(len(y)), y):
        sss = StratifiedShuffleSplit(n_splits=1, test_size=CFG["FRACAO_PARADA"],
                                     random_state=semente + len(dobras))
        i_aj, i_par = next(sss.split(np.zeros(len(itr)), y[itr]))
        dobras.append(dict(ajuste=itr[i_aj], parada=itr[i_par], validacao=iva))
    return dobras

DOBRAS_POR_SEMENTE = {s: constroi_dobras(s) for s in CFG["SEMENTES"]}
_d0 = DOBRAS_POR_SEMENTE[SEMENTE_MESTRA][0]
print(f"\n  Por dobra: ajuste {len(_d0['ajuste'])}, parada {len(_d0['parada'])}, "
      f"validação {len(_d0['validacao'])}")
print(f"  Proporção de casos — ajuste {Y_DEV[_d0['ajuste']].mean():.3f}, "
      f"parada {Y_DEV[_d0['parada']].mean():.3f}, "
      f"validação {Y_DEV[_d0['validacao']].mean():.3f}")

# -----------------------------------------------------------------------------
# QUANTO SINAL EXISTE POR FRAGMENTO, E QUANTO POR PACIENTE
#
# O «d de Cohen» mede a distância entre dois grupos em unidades de desvio
# padrão: 0,2 é um efeito pequeno, 0,8 é grande. Calculamos duas vezes 
# tratando cada FRAGMENTO como observação, e cada PACIENTE (a média dos seus
# 5.000) como observação, usando exatamente os mesmos pacientes nos dois
# níveis. É a medida que justifica a existência do Multiple Instance Learning:
# nenhum fragmento sozinho decide.
# -----------------------------------------------------------------------------
print("\n" + "-" * 96)
print("QUANTO SINAL EXISTE POR FRAGMENTO E POR PACIENTE")
print("-" * 96)
rng_d = np.random.default_rng(SEMENTE_MESTRA)
casos_loc = np.where(Y_DEV == 1)[0]; ctrl_loc = np.where(Y_DEV == 0)[0]
n_d = min(len(casos_loc), len(ctrl_loc), CFG["N_PACIENTES_POR_CLASSE_D"])
sel_caso = np.sort(rng_d.choice(casos_loc, n_d, replace=False))
sel_ctrl = np.sort(rng_d.choice(ctrl_loc, n_d, replace=False))

def cohen_d(a, b):
    return np.abs(a.mean(0) - b.mean(0)) / (
        np.sqrt((a.var(0, ddof=1) + b.var(0, ddof=1)) / 2) + 1e-12)

d_pac_v = cohen_d(V_CAD[sel_caso], V_CAD[sel_ctrl])
bp = np.asarray(CAD[np.sort(IDX_DEV[sel_caso])], np.float32).reshape(-1, D_EMB)
bn = np.asarray(CAD[np.sort(IDX_DEV[sel_ctrl])], np.float32).reshape(-1, D_EMB)
d_frag_v = cohen_d(bp, bn)
del bp, bn

D_FRAG, D_PAC = float(np.median(d_frag_v)), float(np.median(d_pac_v))
print(f"  Amostra (a mesma nos dois níveis) : {n_d} casos + {n_d} controles")
print(f"  Por fragmento isolado : d = {D_FRAG:.4f}   (mediana das {D_EMB} dimensões)")
print(f"  Por paciente          : d = {D_PAC:.4f}   (mediana das {D_EMB} dimensões)")

# -----------------------------------------------------------------------------
# O ENCODER CONTÉM ALGO ALÉM DE COMPOSIÇÃO E LOCALIZAÇÃO?
#
# Tentamos reconstruir os 256 números do Caduceus a partir de duas coisas
# simples: as contagens de 4-mers (composição de bases) e a ocupação de
# promotores (onde o fragmento caiu). O R² diz quanto da variação reproduzimos:
# R² = 1 é reconstrução perfeita.
#
# O resíduo é o candidato ao que o pré-treino acrescentou. Se
# ele discriminar casos de controles, há sinal além da composição.
#
# O resíduo é reconstruído DENTRO DE CADA RÉPLICA, com as dobras daquela
# réplica: «cinco réplicas independentes» seria falso se elas partilhassem a
# construção da feature.
# -----------------------------------------------------------------------------
print("\n" + "=" * 96)
print("O ENCODER CONTÉM ALGO ALÉM DE COMPOSIÇÃO E LOCALIZAÇÃO?")
print("=" * 96)

def constroi_residuo(dobras, com_pca_padronizada=False):
    """Devolve o resíduo fora-da-dobra e os R² de cada reconstrução."""
    resid = np.zeros_like(V_CAD)
    r2 = dict(kmer=[], ocup=[], ambos=[], ocup_pad=[]); alphas = []
    for dobra in dobras:
        tr, va = dobra["ajuste"], dobra["validacao"]
        pca = PCA(n_components=CFG["N_COMPONENTES_OCUP"], svd_solver="randomized",
                  random_state=SEMENTE_MESTRA).fit(V_OCUP[tr])
        Z = pca.transform(V_OCUP)
        blocos = [("kmer", V_KMER), ("ocup", Z), ("ambos", np.hstack([V_KMER, Z]))]
        if com_pca_padronizada:
            pp = Pipeline([("esc", StandardScaler()),
                           ("pca", PCA(n_components=CFG["N_COMPONENTES_OCUP"],
                                       svd_solver="randomized",
                                       random_state=SEMENTE_MESTRA))]).fit(V_OCUP[tr])
            blocos.append(("ocup_pad", pp.transform(V_OCUP)))
        for etiq, Xe in blocos:
            Xe = np.ascontiguousarray(Xe, dtype=np.float32)
            mod = Pipeline([("esc", StandardScaler()),
                            ("ridge", RidgeCV(alphas=np.logspace(-3, 3, 13)))]).fit(
                                Xe[tr], V_CAD[tr])
            prev = mod.predict(Xe[va])
            r2[etiq].append(r2_score(V_CAD[va], prev, multioutput="variance_weighted"))
            if etiq == "ambos":
                resid[va] = V_CAD[va] - prev
                # Para a coorte externa, o resíduo é a média das cinco dobras.
                resid[N_DEV:] += (V_CAD[N_DEV:] - mod.predict(Xe[N_DEV:])) / len(dobras)
                alphas.append(float(np.ravel(mod.named_steps["ridge"].alpha_)[0]))
    return resid, r2, alphas

t0 = time.time()
RESIDUOS, R2_TODOS = {}, dict(kmer=[], ocup=[], ambos=[], ocup_pad=[])
for i_s, s in enumerate(CFG["SEMENTES"]):
    resid, r2, alphas = constroi_residuo(
        DOBRAS_POR_SEMENTE[s],
        com_pca_padronizada=(i_s == 0 and CFG["VERIFICACOES_EXTRA"]))
    RESIDUOS[s] = resid
    for k in r2:
        R2_TODOS[k].extend(r2[k])
    if i_s == 0:
        ALPHAS_S1 = alphas

R2_KMER  = float(np.mean(R2_TODOS["kmer"]))
R2_OCUP  = float(np.mean(R2_TODOS["ocup"]))
R2_RESID = float(np.mean(R2_TODOS["ambos"]))
R2_OCUP_PAD = float(np.mean(R2_TODOS["ocup_pad"])) if R2_TODOS["ocup_pad"] else float("nan")

print(f"\n  Quanto do vetor do Caduceus é reconstruível (R² fora da dobra, média de")
print(f"  {len(R2_TODOS['ambos'])} dobras = {len(CFG['SEMENTES'])} sementes × {CFG['N_DOBRAS']} dobras):")
print(f"    só composição de 4-mers     : {R2_KMER:.3f}")
print(f"    só ocupação de promotores   : {R2_OCUP:.3f}")
if np.isfinite(R2_OCUP_PAD):
    print(f"      (com PCA padronizada)     : {R2_OCUP_PAD:.3f}")
print(f"    as duas juntas              : {R2_RESID:.3f}   "
      f"[{min(R2_TODOS['ambos']):.3f}–{max(R2_TODOS['ambos']):.3f}]")
print(f"  Regularização escolhida (semente 1): {', '.join(f'{a:.3g}' for a in ALPHAS_S1)}")
print(f"  ({time.time()-t0:.0f} s)")

RESID_OK       = R2_RESID >= CFG["R2_ABORTA_RESIDUO"]
RESID_INTERPRE = R2_RESID >= CFG["R2_INTERPRETA_RESIDUO"]
if not RESID_OK:
    print(f"\n  ANÁLISE DO RESÍDUO ABORTADA: a reconstrução falhou (R² = {R2_RESID:.3f}).")
    print(f"  Nesse caso o resíduo é praticamente o vetor original, e a sua AUROC")
    print(f"  não diz nada sobre o pré-treino.")

# -----------------------------------------------------------------------------
# DISTRIBUIÇÃO NULA EMPÍRICA
#
# Embaralhamos os rótulos e refazemos o procedimento. A AUROC que um modelo
# atinge sobre rótulos aleatórios não é exatamente 0,500: com 537 pacientes e
# 256 dimensões, ela flutua. Esta é a distribuição dessa flutuação.
#
# Três exigências: a nula usa a MESMA representação, o MESMO classificador e a
# MESMA estrutura de replicação (5 sementes × 5 dobras) da estatística observada.
#
# Sobre as partições. Quando a representação NÃO depende da partição, as dobras
# são refeitas nos rótulos permutados (`reestratifica=True`), para que a
# partição não fique condicionada ao desfecho verdadeiro. Quando a representação
# FOI CONSTRUÍDA fora-da-dobra sob certas partições — o caso do resíduo —,
# refazê-las descasaria a feature das dobras e o pareamento fora-da-dobra se
# perderia justamente no nulo, inflando-o. Aí mantemos as originais.
# -----------------------------------------------------------------------------
def distribuicao_nula(X_por_semente, n_perm, reestratifica):
    nulos = []
    rngp = np.random.RandomState(SEMENTE_MESTRA)
    for _ in range(n_perm):
        y = rngp.permutation(Y_DEV)
        por_semente = []
        for s in CFG["SEMENTES"]:
            X = X_por_semente[s]; aucs = []
            dobras = constroi_dobras(s, y=y) if reestratifica else DOBRAS_POR_SEMENTE[s]
            for dobra in dobras:
                tr, va = dobra["ajuste"], dobra["validacao"]
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore", ConvergenceWarning)
                    m = Pipeline([("e", StandardScaler()),
                                  ("l", LogisticRegression(max_iter=1000))]).fit(X[tr], y[tr])
                aucs.append(roc_auc_score(y[va], m.predict_proba(X[va])[:, 1]))
            por_semente.append(np.mean(aucs))
        nulos.append(np.mean(por_semente))
    return np.array(nulos)

print("\n" + "-" * 96)
print(f"DISTRIBUIÇÃO NULA EMPÍRICA DO RESÍDUO ({CFG['N_PERMUTACOES']} embaralhamentos × "
      f"{len(CFG['SEMENTES'])} sementes × {CFG['N_DOBRAS']} dobras)")
print("-" * 96)
t0 = time.time()
if RESID_OK:
    NULOS_RESID = distribuicao_nula(RESIDUOS, CFG["N_PERMUTACOES"], reestratifica=False)
    PERC95_RESID = float(np.percentile(NULOS_RESID, 95))
    print(f"  Mediana {np.median(NULOS_RESID):.3f} · percentil 95 {PERC95_RESID:.3f}   "
          f"({time.time()-t0:.0f} s)")
else:
    NULOS_RESID, PERC95_RESID = None, float("nan")

if CFG["VERIFICACOES_EXTRA"]:
    t0 = time.time()
    NULOS_CAD = distribuicao_nula({s: V_CAD for s in CFG["SEMENTES"]},
                                  CFG["N_PERMUTACOES"], reestratifica=True)
    PERC95_CAD = float(np.percentile(NULOS_CAD, 95))
    print(f"  [extra] Vetor completo do Caduceus — mediana {np.median(NULOS_CAD):.3f} · "
          f"percentil 95 {PERC95_CAD:.3f}   ({time.time()-t0:.0f} s)")
else:
    PERC95_CAD = float("nan")

# -----------------------------------------------------------------------------
print("\n" + "-" * 96)
print("LIMITES DO QUE ESTA CÉLULA MEDE")
print("-" * 96)
print("  • O resíduo depende da classe de reconstrução escolhida aqui, linear.")
print("    Ausência de sinal sob um modelo linear não prova ausência de informação.")
print("  • A distribuição nula também é linear: não cobre estrutura não linear,")
print("    nem o patamar mais alto de redes com atenção e seleção de época.")
print(f"  • Com {CFG['N_PERMUTACOES']} permutações, o menor p atingível é "
      f"{1/(CFG['N_PERMUTACOES']+1):.4f}.")
print("  • O nulo do resíduo usa as partições que o geraram. Refazê-las descasaria")
print("    a feature das dobras e inflaria o nulo — a alternativa seria pior.")
print("  • O d de Cohen mediano por dimensão é uma descrição do regime, não uma")
print("    avaliação preditiva por fragmento: não exclui dimensões informativas.")

print("\n" + "=" * 96); print("FIM DA CÉLULA 2"); print("=" * 96)

# =============================================================================
# ===== FIM CÉLULA 2 ==========================================================
# =============================================================================

CÉLULA 2 — REPRESENTAÇÕES E CONTEÚDO DO ENCODER

  Desenvolvimento : 537 pacientes (276 casos / 261 controles)
  Externa         : 225 pacientes
  Carcinoma hepatocelular no treino : 0
  As duas tarefas diferem: treino pan-câncer, avaliação num tipo tumoral
  ausente do treino, noutra população e noutro protocolo.

  Construindo vetores por paciente...
  concluído em 15 s
  Fragmentos fora do histograma de comprimento : 0.0000%
  Zeros na matriz de ocupação de promotores    : 88.4%

  Por dobra: ajuste 364, parada 65, validação 108
  Proporção de casos — ajuste 0.514, parada 0.508, validação 0.519

------------------------------------------------------------------------------------------------
QUANTO SINAL EXISTE POR FRAGMENTO E POR PACIENTE
------------------------------------------------------------------------------------------------
  Amostra (a mesma nos dois níveis) : 60 casos + 60 controles
  Por fragmento isolado : d = 0.0104   (mediana das 256 dimensões)
  Por paciente        

In [3]:
# =============================================================================
# ===== CÉLULA 3 — A ESCADA DE SEIS DEGRAUS ===================================
# =============================================================================

print("=" * 96)
print("CÉLULA 3 — A ESCADA DE SEIS DEGRAUS, REPETIDA EM CINCO SEMENTES")
print("=" * 96)

AGREGACAO = {"cad_att": "atencao", "cad_mean": "media_apos_projecao",
             "cad_vec_mlp": "media_antes_projecao", "cad_vec_mlpdir": "media_sem_projecao",
             "kmer_att": "atencao", "kmer_mean": "media_apos_projecao",
             "kmer_vec_mlp": "media_antes_projecao", "length": "atencao"}
FONTE = {"cad_att": "caduceus", "cad_mean": "caduceus", "cad_vec_mlp": "caduceus",
         "cad_vec_mlpdir": "caduceus", "kmer_att": "kmers", "kmer_mean": "kmers",
         "kmer_vec_mlp": "kmers", "length": "zeros", "delfi": "tabular"}
# Nestes modos o modelo nunca vê fragmentos individuais, só a média. Como a
# padronização é afim, média((x-μ)/σ) = (média(x)-μ)/σ, e a entrada pode ser
# derivada dos vetores já calculados. Um gate verifica essa igualdade.
SEM_BOLSA = {"media_antes_projecao", "media_sem_projecao"}
NEURAIS = ["cad_att", "cad_mean", "cad_vec_mlp", "cad_vec_mlpdir",
           "kmer_att", "kmer_mean", "kmer_vec_mlp", "length", "delfi"]
LINEARES = [("cad_lin", "V_CAD"), ("cad_vec_lin", "X_CAD_COMP"),
            ("rep_kmer", "V_KMER"), ("rep_len", "V_HIST"), ("rep_delfi", "V_DELFI")]
X_CAD_COMP = np.hstack([V_CAD, V_COMP]).astype(np.float32)
MATRIZ = {"V_CAD": V_CAD, "X_CAD_COMP": X_CAD_COMP, "V_KMER": V_KMER,
          "V_HIST": V_HIST, "V_DELFI": V_DELFI}


def ram_disponivel():
    try:
        with open("/proc/meminfo") as f:
            for linha in f:
                if linha.startswith("MemAvailable:"):
                    return int(linha.split()[1]) * 1024
    except Exception:
        pass
    return None

def sobe_em_blocos(mm, lote):
    dtype_torch = torch.from_numpy(np.empty(0, dtype=mm.dtype)).dtype
    t = torch.empty(tuple(mm.shape), dtype=dtype_torch, device=DISPOSITIVO)
    for i in range(0, mm.shape[0], lote):
        t[i:i+lote] = torch.from_numpy(np.ascontiguousarray(mm[i:i+lote])).to(DISPOSITIVO)
    return t

BOLSA_GPU = {}
LOTE_SUBIDA = 50
if CFG["BOLSAS_NA_GPU"] and torch.cuda.is_available():
    livre, _ = torch.cuda.mem_get_info()
    preciso = int((CAD.nbytes + KMER.nbytes + COMP.nbytes * 2) * 1.05)
    ram = ram_disponivel()
    pico_host = LOTE_SUBIDA * CAD.shape[1] * CAD.shape[2] * CAD.dtype.itemsize
    print(f"\n  Embeddings: {CAD.dtype}, {CAD.nbytes/2**30:.2f} GB cada · "
          f"GPU precisa {preciso/2**30:.2f} GB de {livre/2**30:.2f} GB livres")
    print(f"  Pico de RAM do host na subida: {pico_host/2**30:.3f} GB" +
          (f" de {ram/2**30:.2f} GB livres" if ram else ""))
    if preciso < livre * 0.60:
        BOLSA_GPU["caduceus"] = sobe_em_blocos(CAD, LOTE_SUBIDA)
        BOLSA_GPU["kmers"]    = sobe_em_blocos(KMER, LOTE_SUBIDA)
        # float32 aqui, igual ao caminho por disco: elimina divergência numérica
        # entre os dois caminhos. O tensor é pequeno.
        BOLSA_GPU["comp"]     = torch.from_numpy(COMP.astype(np.float32)).to(DISPOSITIVO)
        print(f"  Bolsas residentes na GPU.")
    else:
        print(f"  Memória insuficiente; leitura por disco (mais lenta).")
NA_GPU = bool(BOLSA_GPU)

def estatisticas(fonte, idx_ajuste):
    """Média e desvio de cada canal, calculados só no conjunto de treino."""
    s = sq = 0.0; nc = 0
    for i in range(0, len(idx_ajuste), 60):
        c = COMP[np.sort(idx_ajuste[i:i+60])].astype(np.float64)
        s += c.sum(); sq += (c**2).sum(); nc += c.size
    mu_c = s / nc; sd_c = np.sqrt(max(sq/nc - mu_c**2, 1e-12))
    if fonte == "zeros":
        e = dict(mu=np.zeros(D_EMB, np.float32), sigma=np.ones(D_EMB, np.float32),
                 mu_comp=np.float32(mu_c), sigma_comp=np.float32(sd_c))
    elif NA_GPU:
        idx = torch.as_tensor(np.sort(idx_ajuste), device=DISPOSITIVO, dtype=torch.long)
        soma = torch.zeros(D_EMB, dtype=torch.float64, device=DISPOSITIVO)
        somq = torch.zeros(D_EMB, dtype=torch.float64, device=DISPOSITIVO); n = 0
        for i in range(0, len(idx), 60):
            x = BOLSA_GPU[fonte][idx[i:i+60]].double()
            soma += x.sum(dim=(0, 1)); somq += (x**2).sum(dim=(0, 1))
            n += x.shape[0] * x.shape[1]
        mu = (soma/n); sd = (somq/n - mu**2).clamp_min(1e-12).sqrt()
        e = dict(mu=mu.float().cpu().numpy(), sigma=sd.float().cpu().numpy(),
                 mu_comp=np.float32(mu_c), sigma_comp=np.float32(sd_c))
    else:
        M = CAD if fonte == "caduceus" else KMER
        soma = np.zeros(D_EMB); somq = np.zeros(D_EMB); n = 0
        for i in range(0, len(idx_ajuste), 30):
            b = np.asarray(M[np.sort(idx_ajuste[i:i+30])], dtype=np.float64)
            soma += b.sum(axis=(0,1)); somq += (b**2).sum(axis=(0,1))
            n += b.shape[0]*b.shape[1]
        mu = soma/n; sd = np.sqrt(np.maximum(somq/n - mu**2, 1e-12))
        e = dict(mu=mu.astype(np.float32), sigma=sd.astype(np.float32),
                 mu_comp=np.float32(mu_c), sigma_comp=np.float32(sd_c))
    if NA_GPU:
        e["mu_t"]    = torch.as_tensor(e["mu"], device=DISPOSITIVO)
        e["sigma_t"] = torch.as_tensor(e["sigma"], device=DISPOSITIVO)
        e["muc_t"]   = torch.as_tensor(float(e["mu_comp"]), device=DISPOSITIVO)
        e["sdc_t"]   = torch.as_tensor(float(e["sigma_comp"]), device=DISPOSITIVO)
    return e

def monta_bolsa(fonte, idx_glob, est):
    """Bolsa padronizada (lote, 5000, 257) como tensor na GPU."""
    if NA_GPU:
        idx  = torch.as_tensor(idx_glob, device=DISPOSITIVO, dtype=torch.long)
        comp = (BOLSA_GPU["comp"][idx].unsqueeze(-1) - est["muc_t"]) / est["sdc_t"]
        if fonte == "zeros":
            base = torch.zeros(len(idx_glob), CFG["N_FRAGMENTOS"], D_EMB, device=DISPOSITIVO)
        else:
            base = (BOLSA_GPU[fonte][idx].float() - est["mu_t"]) / est["sigma_t"]
        return torch.cat([base, comp], dim=2)
    inv = np.argsort(np.argsort(idx_glob)); srt = np.sort(idx_glob)
    if fonte == "zeros":
        base = np.zeros((len(idx_glob), CFG["N_FRAGMENTOS"], D_EMB), np.float32)
    else:
        M = CAD if fonte == "caduceus" else KMER
        base = ((np.asarray(M[srt], dtype=np.float32)[inv] - est["mu"]) / est["sigma"])
    comp = ((COMP[srt].astype(np.float32)[inv][..., None] - est["mu_comp"])
            / est["sigma_comp"])
    return torch.from_numpy(np.ascontiguousarray(
        np.concatenate([base, comp], axis=2))).to(DISPOSITIVO)

def vetor_medio(fonte, est):
    """O mesmo vetor que a média da bolsa produziria, derivado dos agregados."""
    base = {"caduceus": V_CAD, "kmers": V_KMER}.get(
        fonte, np.zeros((len(ordem), D_EMB), np.float32))
    if fonte != "zeros":
        base = (base - est["mu"]) / est["sigma"]
    comp = (V_COMP - est["mu_comp"]) / est["sigma_comp"]
    return np.ascontiguousarray(np.hstack([base, comp]).astype(np.float32))

class ModeloBolsa(nn.Module):
    """
    Uma arquitetura, quatro modos de combinar os fragmentos:

      atencao               projeta cada fragmento e aprende um peso para cada um
      media_apos_projecao   projeta cada fragmento, depois tira a média
      media_antes_projecao  tira a média primeiro, depois projeta
      media_sem_projecao    tira a média e vai direto para a cabeça

    A projeção e a cabeça nascem antes do ramo de atenção, para que braços com
    o mesmo prefixo partam de pesos idênticos nessas camadas. Isso não vale para
    'media_sem_projecao', cuja cabeça tem outra forma e consome outra quantidade
    de números aleatórios — é outra arquitetura, por construção.
    """
    def __init__(self, d_in, d_oculta, modo):
        super().__init__()
        self.modo = modo
        if modo == "media_sem_projecao":
            self.projecao, d_cab = None, d_in
        else:
            self.projecao = nn.Sequential(nn.Linear(d_in, d_oculta), nn.ReLU())
            d_cab = d_oculta
        self.cabeca = nn.Sequential(
            nn.Linear(d_cab, d_oculta), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(d_oculta, d_oculta // 2), nn.ReLU(),
            nn.Linear(d_oculta // 2, 1))
        self.atencao = nn.Sequential(
            nn.Linear(d_oculta, d_oculta // 2), nn.Tanh(),
            nn.Linear(d_oculta // 2, 1)) if modo == "atencao" else None
        self.ultimos_pesos = None

    def forward(self, x):
        if self.modo == "atencao":
            h = self.projecao(x)
            w = torch.softmax(self.atencao(h), dim=1)
            self.ultimos_pesos = w.detach()
            z = (w * h).sum(dim=1)
        elif self.modo == "media_apos_projecao":
            z = self.projecao(x).mean(dim=1)
        else:
            v = x if x.dim() == 2 else x.mean(dim=1)
            z = v if self.projecao is None else self.projecao(v)
        return self.cabeca(z).squeeze(-1)

def curva_suavizada(trajetoria, janela):
    """
    A AUROC no conjunto de parada flutua mais entre épocas vizinhas do que o
    modelo realmente melhora. Escolher o máximo bruto é escolher ruído.
    Suavizamos com média móvel centrada antes de escolher a época e antes de
    aplicar o limiar da análise de sensibilidade.
    """
    a = np.asarray(trajetoria, float); n = len(a); m = janela // 2
    return np.array([a[max(0, i-m):min(n, i+m+1)].mean() for i in range(n)])

def treina(braco, dobra, est_dobra, semente):
    aj, par, val = dobra["ajuste"], dobra["parada"], dobra["validacao"]
    fonte = FONTE[braco]; modo = AGREGACAO.get(braco)
    est = est_dobra.get(fonte) if CFG["PADRONIZA_BOLSA"] else None
    sem_bolsa = modo in SEM_BOLSA
    V_AG = vetor_medio(fonte, est) if sem_bolsa else None

    torch.manual_seed(semente)
    if braco == "delfi":
        modelo = nn.Sequential(
            nn.Linear(DELFI.shape[1], CFG["DIM_OCULTA"]), nn.ReLU(), nn.Dropout(0.3),
            nn.Linear(CFG["DIM_OCULTA"], CFG["DIM_OCULTA"]//2), nn.ReLU(),
            nn.Linear(CFG["DIM_OCULTA"]//2, 1)).to(DISPOSITIVO)
        esc = StandardScaler().fit(DELFI[IDX_DEV[aj]])
    else:
        modelo = ModeloBolsa(D_EMB + 1, CFG["DIM_OCULTA"], modo).to(DISPOSITIVO)
        esc = None
    otim = torch.optim.AdamW(modelo.parameters(), lr=CFG["TAXA_APRENDIZADO"],
                             weight_decay=CFG["DECAIMENTO_PESO"])
    perda_fn = nn.BCEWithLogitsLoss()

    def passa(idx):
        if braco == "delfi":
            x = np.ascontiguousarray(esc.transform(DELFI[idx]), dtype=np.float32)
            return modelo(torch.from_numpy(x).to(DISPOSITIVO)).squeeze(-1)
        ent = (torch.from_numpy(np.ascontiguousarray(V_AG[POS[idx]])).to(DISPOSITIVO)
               if sem_bolsa else monta_bolsa(fonte, idx, est))
        return modelo(ent)

    y_aj, y_par = Y_DEV[aj], Y_DEV[par]
    rngl = np.random.RandomState(semente)
    trajetoria, checkpoints = [], []
    melhor, sem_melhora = -1.0, 0
    n_cortes = n_passos = 0

    for epoca in range(1, CFG["MAX_EPOCAS"] + 1):
        modelo.train()
        perm = rngl.permutation(len(aj))
        for i in range(0, len(perm), CFG["TAMANHO_LOTE"]):
            sub = perm[i:i+CFG["TAMANHO_LOTE"]]
            logits = passa(IDX_DEV[aj[sub]])
            alvo = torch.from_numpy(np.ascontiguousarray(
                y_aj[sub], dtype=np.float32)).to(DISPOSITIVO)
            otim.zero_grad(set_to_none=True)
            perda_fn(logits, alvo).backward()
            nrm = float(nn.utils.clip_grad_norm_(modelo.parameters(), CFG["CLIP_GRADIENTE"]))
            n_passos += 1; n_cortes += int(nrm > CFG["CLIP_GRADIENTE"])
            otim.step()
        modelo.eval()
        with torch.no_grad():
            p = [torch.sigmoid(passa(IDX_DEV[par[i:i+CFG["TAMANHO_LOTE"]]])).cpu().numpy()
                 for i in range(0, len(par), CFG["TAMANHO_LOTE"])]
            a_par = roc_auc_score(y_par, np.concatenate(p))
        trajetoria.append(a_par)
        # Guardamos todas as épocas porque a escolha final usa a curva suavizada,
        # que só existe depois que o treino termina.
        checkpoints.append({k: v.detach().cpu().clone()
                            for k, v in modelo.state_dict().items()})
        if a_par > melhor + 1e-5:
            melhor, sem_melhora = a_par, 0
        else:
            sem_melhora += 1
            if epoca >= CFG["MIN_EPOCAS"] and sem_melhora >= CFG["PACIENCIA"]:
                break

    suave = curva_suavizada(trajetoria, CFG["JANELA_SUAVIZACAO"])
    i_suave, i_cru = int(np.argmax(suave)), int(np.argmax(trajetoria))
    modelo.load_state_dict(checkpoints[i_suave]); modelo.eval()
    del checkpoints

    def infere(idx_glob):
        out, pesos = [], []
        with torch.no_grad():
            for i in range(0, len(idx_glob), CFG["TAMANHO_LOTE"]):
                lote = idx_glob[i:i+CFG["TAMANHO_LOTE"]]
                out.append(torch.sigmoid(passa(lote)).cpu().numpy())
                if modo == "atencao" and modelo.ultimos_pesos is not None:
                    pesos.append(modelo.ultimos_pesos.squeeze(-1).cpu().numpy())
        return np.concatenate(out), (np.concatenate(pesos) if pesos else None)

    p_val, _ = infere(IDX_DEV[val])
    p_ext, w_ext = infere(IDX_EXT)
    del modelo; torch.cuda.empty_cache()
    return p_val, p_ext, w_ext, dict(
        epoca=i_suave + 1, epoca_crua=i_cru + 1, n_epocas=len(trajetoria),
        pico_parada_suave=float(np.max(suave)),
        pico_parada_cru=float(np.max(trajetoria)),
        cortes=n_cortes, passos=n_passos)

def ajusta_logistica(X, y, padroniza=True):
    passos = ([("esc", StandardScaler())] if padroniza else []) + \
             [("log", LogisticRegression(max_iter=3000, C=1.0))]
    with warnings.catch_warnings(record=True) as cap:
        warnings.simplefilter("always", ConvergenceWarning)
        m = Pipeline(passos).fit(X, y)
    return m, sum(1 for w in cap if issubclass(w.category, ConvergenceWarning))

# -----------------------------------------------------------------------------
# Uma réplica = um sorteio completo: novas partições, novas inicializações, e o
# resíduo reconstruído com as dobras desta réplica.
# -----------------------------------------------------------------------------
def executa_replica(semente, verbose=True):
    dobras = DOBRAS_POR_SEMENTE[semente]
    residuo_desta_replica = RESIDUOS[semente] if RESID_OK else None

    res = {k: dict(auc_dobras=[], oof=np.full(N_DEV, np.nan), ext=[])
           for k, _ in LINEARES}
    res["cad_vec_lin_frag"] = dict(auc_dobras=[], oof=np.full(N_DEV, np.nan), ext=[])
    for b in NEURAIS:
        res[b] = dict(auc_dobras=[], oof=np.full(N_DEV, np.nan), ext=[],
                      epocas=[], epocas_cruas=[], pico_suave=[], pico_cru=[],
                      cortes=0, passos=0, pesos=[])
    if RESID_OK:
        res["rep_cad_resid"] = dict(auc_dobras=[], oof=np.full(N_DEV, np.nan), ext=[])
    avisos = 0

    for i_d, dobra in enumerate(dobras, start=1):
        tr, va = dobra["ajuste"], dobra["validacao"]
        est_dobra = {f: estatisticas(f, IDX_DEV[tr]) for f in ("caduceus", "kmers", "zeros")}

        # --- braços lineares: padronização entre pacientes ---
        entradas = [(k, MATRIZ[m]) for k, m in LINEARES]
        if RESID_OK:
            entradas.append(("rep_cad_resid", residuo_desta_replica))
        for chave, X in entradas:
            m, nw = ajusta_logistica(X[tr], Y_DEV[tr]); avisos += nw
            pv = m.predict_proba(X[va])[:, 1]
            res[chave]["oof"][va] = pv
            res[chave]["auc_dobras"].append(roc_auc_score(Y_DEV[va], pv))
            res[chave]["ext"].append(m.predict_proba(X[N_DEV:])[:, 1])

        # --- o mesmo vetor, na escala de fragmento que a rede recebe ---
        Xf = vetor_medio("caduceus", est_dobra["caduceus"])
        m, nw = ajusta_logistica(Xf[tr], Y_DEV[tr], padroniza=False); avisos += nw
        pv = m.predict_proba(Xf[va])[:, 1]
        res["cad_vec_lin_frag"]["oof"][va] = pv
        res["cad_vec_lin_frag"]["auc_dobras"].append(roc_auc_score(Y_DEV[va], pv))
        res["cad_vec_lin_frag"]["ext"].append(m.predict_proba(Xf[N_DEV:])[:, 1])

        # --- braços neurais ---
        for b in NEURAIS:
            t0 = time.time()
            pv, pe, we, diag = treina(b, dobra, est_dobra, semente + 1000 * i_d)
            res[b]["oof"][va] = pv
            res[b]["auc_dobras"].append(roc_auc_score(Y_DEV[va], pv))
            res[b]["ext"].append(pe)
            res[b]["epocas"].append(diag["epoca"])
            res[b]["epocas_cruas"].append(diag["epoca_crua"])
            res[b]["pico_suave"].append(diag["pico_parada_suave"])
            res[b]["pico_cru"].append(diag["pico_parada_cru"])
            res[b]["cortes"] += diag["cortes"]; res[b]["passos"] += diag["passos"]
            if we is not None:
                res[b]["pesos"].append(we)
            if verbose:
                marca = ("" if diag["pico_parada_suave"] >= CFG["LIMIAR_PARADA"]
                         else "  [abaixo do limiar de parada]")
                print(f"    {nome(b):<62s} {res[b]['auc_dobras'][-1]:.3f}  "
                      f"ép {diag['epoca']:>2d}  ({time.time()-t0:.0f}s){marca}")

    for k in res:
        res[k]["auc_dobras"] = np.array(res[k]["auc_dobras"])
        res[k]["media"] = float(res[k]["auc_dobras"].mean())
        res[k]["externo"] = np.mean(res[k]["ext"], axis=0)
    return res, avisos

REPLICAS, AVISOS_CONV = [], 0
for i_s, semente in enumerate(CFG["SEMENTES"], start=1):
    print(f"\n{'='*96}\nRÉPLICA {i_s} de {len(CFG['SEMENTES'])} — semente {semente}\n{'='*96}")
    t0 = time.time()
    r, nw = executa_replica(semente); AVISOS_CONV += nw
    REPLICAS.append(r)
    print(f"  Réplica concluída em {(time.time()-t0)/60:.1f} min")

CHAVES = list(REPLICAS[0].keys())
# Escore ensemble por paciente: média das cinco réplicas. É o mesmo objeto usado
# pela escada, pelo ponto de operação e pela avaliação interna agregada.
OOF_ENS = {k: np.mean([r[k]["oof"] for r in REPLICAS], axis=0) for k in CHAVES}

# -----------------------------------------------------------------------------
# GATES numéricos
# -----------------------------------------------------------------------------
print("\n" + "-" * 96)
print("GATE — O VETOR PRÉ-CALCULADO É IGUAL À MÉDIA DA BOLSA?")
print("-" * 96)
est_v = {f: estatisticas(f, IDX_DEV[:300]) for f in ("caduceus", "kmers", "zeros")}
amostra = IDX_DEV[:8]; ERROS = {}
for f in ("caduceus", "kmers"):
    a1 = monta_bolsa(f, amostra, est_v[f]).mean(dim=1).cpu().numpy()
    a2 = vetor_medio(f, est_v[f])[POS[amostra]]
    ERROS[f] = float(np.abs(a1 - a2).max())
    print(f"  {f:<10s} diferença máxima : {ERROS[f]:.2e}")
    if ERROS[f] > 1e-4:
        raise RuntimeError(f"Os dois caminhos divergem para '{f}'; o degrau da "
                           f"cabeça não isolaria a cabeça.")
dp_nulo = float(monta_bolsa("zeros", amostra, est_v["zeros"])[:, :, -1].std().cpu())
print(f"  Controle nulo, desvio do canal de comprimento : {dp_nulo:.3f} (esperado ~1,0)")
if not (0.5 < dp_nulo < 2.0):
    raise RuntimeError("O canal de comprimento do controle nulo não está padronizado.")
print("  Aprovado.")

# -----------------------------------------------------------------------------
# GATE — A CAMADA LINEAR COMUTA COM A MÉDIA?
#
#
#     Linear(média(x)) = W·média(x) + b = média(W·x + b) = média(Linear(x))
#
# A identidade vale para QUAISQUER pesos. Logo as duas ARQUITETURAS diferem
# exclusivamente na posição da ReLU.
# -----------------------------------------------------------------------------
print("\n" + "-" * 96)
print("GATE — A CAMADA LINEAR COMUTA COM A MÉDIA?")
print("-" * 96)
torch.manual_seed(SEMENTE_MESTRA)
_lin = nn.Linear(D_EMB + 1, CFG["DIM_OCULTA"]).to(DISPOSITIVO)
with torch.no_grad():
    _bolsa = monta_bolsa("caduceus", amostra, est_v["caduceus"])
    ERRO_COMUTA = float((_lin(_bolsa.mean(dim=1)) - _lin(_bolsa).mean(dim=1)).abs().max())
del _bolsa, _lin
print(f"  |Linear(média(x)) − média(Linear(x))| máximo : {ERRO_COMUTA:.2e}")
if ERRO_COMUTA > 1e-3:
    raise RuntimeError("A camada afim não comuta com a média; as duas arquiteturas "
                       "do degrau ORDEM difeririam em mais de uma coisa.")
print(f"  Aprovado. As arquiteturas 'Média→Projeção' e 'Projeção→Média' diferem")
print(f"  exclusivamente na posição da ReLU.")

# -----------------------------------------------------------------------------
# A ESCADA
#
# Cada degrau muda exatamente uma coisa. A soma dos seis iguala a diferença
# entre os extremos por construção aritmética — o que verifica a integridade do
# cálculo, não estabelece causa.
# -----------------------------------------------------------------------------
ESCADA = [
    ("cad_lin",          "cad_vec_lin",      "COMPRIMENTO",  "acrescenta o comprimento do fragmento"),
    ("cad_vec_lin",      "cad_vec_lin_frag", "NORMALIZAÇÃO", "padronização por paciente → por fragmento"),
    ("cad_vec_lin_frag", "cad_vec_mlpdir",   "CABEÇA",       "troca a regressão logística por uma rede"),
    ("cad_vec_mlpdir",   "cad_vec_mlp",      "PROJEÇÃO",     "acrescenta uma projeção DEPOIS da média"),
    ("cad_vec_mlp",      "cad_mean",         "ORDEM",        "move a mesma projeção para ANTES da média"),
    ("cad_mean",         "cad_att",          "AGREGADOR",    "troca a média por atenção aprendida"),
]

print("\n" + "=" * 96)
print("A ESCADA DE SEIS DEGRAUS")
print("=" * 96)

def atingiu_limiar(res_braco, i_fold):
    """A AUROC suavizada no conjunto de parada superou o limiar em alguma época.
    É uma triagem de desempenho mínimo, não um teste de convergência."""
    if "pico_suave" not in res_braco:
        return True                      # braços lineares não têm curva de parada
    return res_braco["pico_suave"][i_fold] >= CFG["LIMIAR_PARADA"]

def bootstrap_delta_auc(y, p_a, p_b, n_boot, semente):
    """Bootstrap pareado de pacientes, com os dois modelos fixos."""
    rng = np.random.default_rng(semente); n = len(y); d = []
    for _ in range(n_boot):
        i = rng.integers(0, n, n)
        if np.unique(y[i]).size < 2:
            continue
        d.append(roc_auc_score(y[i], p_b[i]) - roc_auc_score(y[i], p_a[i]))
    d = np.asarray(d)
    return (float(d.mean()), float(np.percentile(d, 2.5)), float(np.percentile(d, 97.5)))

t0 = time.time()
degraus = []
for orig, dest, etiq, desc in ESCADA:
    por_semente = np.array([r[dest]["media"] - r[orig]["media"] for r in REPLICAS])
    por_dobra, ok = [], []
    for r in REPLICAS:
        for i in range(CFG["N_DOBRAS"]):
            por_dobra.append(r[dest]["auc_dobras"][i] - r[orig]["auc_dobras"][i])
            ok.append(atingiu_limiar(r[orig], i) and atingiu_limiar(r[dest], i))
    por_dobra, ok = np.array(por_dobra), np.array(ok)
    bmed, blo, bhi = bootstrap_delta_auc(Y_DEV, OOF_ENS[orig], OOF_ENS[dest],
                                         CFG["N_BOOTSTRAP"], SEMENTE_MESTRA)
    degraus.append(dict(
        origem=orig, destino=dest, etiqueta=etiq, descricao=desc,
        delta=float(por_semente.mean()), dp_sementes=float(por_semente.std(ddof=1)),
        boot_delta=bmed, boot_inf=blo, boot_sup=bhi,
        neg=int((por_dobra < 0).sum()), n_comp=len(por_dobra), n_ok=int(ok.sum()),
        delta_ok=float(por_dobra[ok].mean()) if ok.any() else float("nan"),
        neg_ok=int((por_dobra[ok] < 0).sum()) if ok.any() else 0,
        delta_falha=float(por_dobra[~ok].mean()) if (~ok).any() else float("nan"),
        neg_falha=int((por_dobra[~ok] < 0).sum()) if (~ok).any() else 0))

soma = sum(d["delta"] for d in degraus)
extremos = float(np.mean([r["cad_att"]["media"] - r["cad_lin"]["media"] for r in REPLICAS]))
discrep = abs(soma - extremos)
print(f"  Soma dos seis degraus       : {soma:+.4f}")
print(f"  Diferença entre os extremos : {extremos:+.4f}")
print(f"  Discrepância                : {discrep:.2e}   (bootstrap: {time.time()-t0:.0f} s)")
if discrep > 1e-9:
    raise RuntimeError("A escada não fecha: erro de indexação ou de agregação.")
print(f"  Aprovado: a decomposição é aritmeticamente exata.\n")

print(f"  {'Degrau':<14s} {'O que muda':<45s} {'Δ':>8s} {'±dp':>7s} {'Sinal':>11s}")
print("  " + "-" * 92)
pior = min(degraus, key=lambda d: d["delta"])
for d in degraus:
    marca = "  ← MAIOR PERDA" if d is pior else ""
    print(f"  {d['etiqueta']:<14s} {d['descricao']:<45s} "
          f"{d['delta']:+8.3f} {d['dp_sementes']:7.3f} "
          f"{d['neg']:>4d}/{d['n_comp']} neg{marca}")

print(f"\n  DOIS ESTIMADORES DO MESMO DEGRAU — não são a mesma quantidade medida duas")
print(f"  vezes. O primeiro é a média das diferenças dobra a dobra, e é o que a")
print(f"  escada soma. O segundo é a diferença entre os escores agrupados das")
print(f"  cinco réplicas. Divergem quando as réplicas discordam entre si.")
print("  " + "-" * 92)
print(f"  {'Degrau':<14s} {'média das diferenças por dobra':>32s}   "
      f"{'diferença do escore agrupado':>38s}")
print(f"  {'':14s} {'± dp entre sementes':>32s}   {'[IC 95% bootstrap pareado]':>38s}")
print(f"  {'':14s} {'(pacientes fixos)':>32s}   {'(modelos fixos)':>38s}")
print("  " + "-" * 92)
for d in degraus:
    e1 = f"{d['delta']:+.3f} ± {d['dp_sementes']:.3f}"
    e2 = f"{d['boot_delta']:+.3f}  [{d['boot_inf']:+.3f}, {d['boot_sup']:+.3f}]"
    cruza = "" if (d["boot_inf"] * d["boot_sup"]) > 0 else "  IC cruza zero"
    print(f"  {d['etiqueta']:<14s} {e1:>32s}   {e2:>38s}{cruza}")
print(f"\n  O bootstrap reamostra os mesmos {N_DEV} pacientes {CFG['N_BOOTSTRAP']} vezes.")
print(f"  'Sinal' conta em quantas das {degraus[0]['n_comp']} comparações "
      f"(semente × dobra) o degrau foi negativo.")

# -----------------------------------------------------------------------------
# Análise de sensibilidade à otimização.
#
# Um degrau pode ficar grande porque o modelo de destino é pior, ou porque ele
# não conseguiu treinar. Esta tabela tenta separar as duas.
# -----------------------------------------------------------------------------
N_EXEC_ABAIXO = int(sum(1 for r in REPLICAS for b in NEURAIS
                        for a in r[b]["pico_suave"] if a < CFG["LIMIAR_PARADA"]))
N_COMP_AFETADAS = sum(d["n_comp"] - d["n_ok"] for d in degraus)

print("\n" + "-" * 96)
print("ANÁLISE DE SENSIBILIDADE À OTIMIZAÇÃO")
print(f"Marca as execuções cuja AUROC suavizada no conjunto de parada superou "
      f"{CFG['LIMIAR_PARADA']:.2f}.")
print("-" * 96)
print(f"  {'Degrau':<14s} {'Atingiram':>11s} {'Δ (atingiram)':>22s} "
      f"{'Δ (não atingiram)':>22s} {'Δ (todas)':>11s}")
print("  " + "-" * 86)
for d in degraus:
    c = (f"{d['delta_ok']:+.3f} ({d['neg_ok']}/{d['n_ok']} neg)"
         if np.isfinite(d["delta_ok"]) else "—")
    nf = d["n_comp"] - d["n_ok"]
    f_ = (f"{d['delta_falha']:+.3f} ({d['neg_falha']}/{nf} neg)"
          if np.isfinite(d["delta_falha"]) else "—")
    print(f"  {d['etiqueta']:<14s} {d['n_ok']:>6d}/{d['n_comp']:<3d} "
          f"{c:>22s} {f_:>22s} {d['delta']:+11.3f}")
print(f"\n  Execuções abaixo do limiar : {N_EXEC_ABAIXO} de "
      f"{len(NEURAIS)*len(CFG['SEMENTES'])*CFG['N_DOBRAS']}")
if N_COMP_AFETADAS == 0:
    print(f"  TODAS as execuções atingiram o limiar. A tentativa de separar falha de")
    print(f"  otimização de efeito arquitetural NÃO foi conclusiva nesta execução:")
    print(f"  vale apenas o resultado sobre as {degraus[0]['n_comp']} comparações.")

# -----------------------------------------------------------------------------
# O mesmo contraste noutra representação. Se um degrau se repete com o mesmo
# sinal em 4-mers, ele não é propriedade do Foundation Model.
# -----------------------------------------------------------------------------
print("\n" + "-" * 96)
print("OS MESMOS DEGRAUS, MEDIDOS EM 4-MERS")
print("-" * 96)
replicacao = []
for etiq, oc, dc, ok_, dk in [
        ("ORDEM", "cad_vec_mlp", "cad_mean", "kmer_vec_mlp", "kmer_mean"),
        ("AGREGADOR", "cad_mean", "cad_att", "kmer_mean", "kmer_att")]:
    pc = np.array([r[dc]["media"] - r[oc]["media"] for r in REPLICAS])
    pk = np.array([r[dk]["media"] - r[ok_]["media"] for r in REPLICAS])
    dif_k = np.concatenate([r[dk]["auc_dobras"] - r[ok_]["auc_dobras"] for r in REPLICAS])
    bk, bklo, bkhi = bootstrap_delta_auc(Y_DEV, OOF_ENS[ok_], OOF_ENS[dk],
                                         CFG["N_BOOTSTRAP"], SEMENTE_MESTRA)
    replicacao.append(dict(degrau=etiq, caduceus=float(pc.mean()), dp_cad=float(pc.std(ddof=1)),
                           kmers=float(pk.mean()), dp_kmer=float(pk.std(ddof=1)),
                           boot_kmer=bk, boot_kmer_inf=bklo, boot_kmer_sup=bkhi,
                           neg_kmer=int((dif_k < 0).sum()), n_kmer=len(dif_k),
                           mesmo_sinal=bool(np.sign(pc.mean()) == np.sign(pk.mean()))))
    print(f"  {etiq:<11s} Caduceus {pc.mean():+.3f} ± {pc.std(ddof=1):.3f}   "
          f"4-mers {pk.mean():+.3f} ± {pk.std(ddof=1):.3f} "
          f"[IC {bklo:+.3f}, {bkhi:+.3f}]  ({int((dif_k < 0).sum())}/{len(dif_k)} neg)   "
          f"{'mesmo sinal' if np.sign(pc.mean())==np.sign(pk.mean()) else 'SINAIS OPOSTOS'}")

# --- controles de otimização -------------------------------------------------
tab_clip = pd.DataFrame([dict(
    Braco=nome(b),
    Taxa_recorte_gradiente=sum(r[b]["cortes"] for r in REPLICAS) /
                           max(sum(r[b]["passos"] for r in REPLICAS), 1),
    Epocas_medias=float(np.mean([e for r in REPLICAS for e in r[b]["epocas"]])),
    Pico_parada_bruto=float(np.median([a for r in REPLICAS for a in r[b]["pico_cru"]])),
    Pico_parada_suave=float(np.median([a for r in REPLICAS for a in r[b]["pico_suave"]])),
    Abaixo_do_limiar=int(sum(1 for r in REPLICAS for a in r[b]["pico_suave"]
                             if a < CFG["LIMIAR_PARADA"])))
    for b in NEURAIS]).sort_values("Abaixo_do_limiar", ascending=False)
TAXA_CLIP = float(tab_clip["Taxa_recorte_gradiente"].max())
INFLACAO_PICO = float((tab_clip["Pico_parada_bruto"] - tab_clip["Pico_parada_suave"]).mean())
print("\n" + "-" * 96)
print("CONTROLES DE OTIMIZAÇÃO")
print("A diferença entre o pico bruto e o suavizado é a inflação por tomar o máximo")
print("de muitas avaliações ruidosas. A triagem usa o suavizado.")
print("-" * 96)
print(tab_clip.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print(f"\n  Inflação média do pico bruto sobre o suavizado: {INFLACAO_PICO:+.3f} de AUROC.")

# -----------------------------------------------------------------------------
print("\n" + "-" * 96)
print("LIMITES DO QUE ESTA CÉLULA MEDE")
print("-" * 96)
print("  • A escada depende do caminho: outra ordem dos degraus daria outras fatias")
print("    com a mesma soma. A identidade telescópica verifica o cálculo, não a causa.")
print(f"  • As {degraus[0]['n_comp']} comparações não são independentes — os conjuntos de treino se")
print("    sobrepõem e os pacientes se repetem. A contagem de sinais é descritiva.")
print("  • NORMALIZAÇÃO muda a referência de padronização E a força efetiva da")
print("    penalização L2, juntas. Não isola escala pura. Mas a dependência de escala")
print("    da penalização é propriedade real de qualquer classificador regularizado.")
print("  • O limiar de parada não mede convergência do otimizador. Tomar o máximo de")
print(f"    dezenas de avaliações num conjunto de ~{len(_d0['parada'])} pacientes cruza qualquer limiar")
print("    fixo com frequência apreciável. A suavização reduz a inflação, não a elimina.")
print("  • O degrau ORDEM mede a consequência de ponta a ponta da escolha")
print("    arquitetural, incluindo o efeito sobre a otimização. Os dois braços são")
print("    treinados separadamente, isso é consequência da escolha, não confundimento.")

print("\n" + "=" * 96); print("FIM DA CÉLULA 3"); print("=" * 96)

# =============================================================================
# ===== FIM CÉLULA 3 ==========================================================
# =============================================================================

CÉLULA 3 — A ESCADA DE SEIS DEGRAUS, REPETIDA EM CINCO SEMENTES

  Embeddings: float16, 1.82 GB cada · GPU precisa 3.83 GB de 14.46 GB livres
  Pico de RAM do host na subida: 0.119 GB de 29.46 GB livres


/tmp/ipykernel_23/2844311531.py:43: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:213.)
  t[i:i+lote] = torch.from_numpy(np.ascontiguousarray(mm[i:i+lote])).to(DISPOSITIVO)


  Bolsas residentes na GPU.

RÉPLICA 1 de 5 — semente 1807
    Caduceus + Comprimento (Projeção → Atenção MIL)                0.751  ép 19  (13s)
    Caduceus + Comprimento (Projeção → Média → MLP)                0.743  ép 29  (9s)
    Caduceus + Comprimento (Média → Projeção → MLP)                0.813  ép 21  (2s)
    Caduceus + Comprimento (Média → MLP)                           0.833  ép 47  (3s)
    4-mers + Comprimento (Projeção → Atenção MIL)                  0.797  ép 12  (6s)
    4-mers + Comprimento (Projeção → Média → MLP)                  0.776  ép 28  (8s)
    4-mers + Comprimento (Média → Projeção → MLP)                  0.816  ép 47  (3s)
    Somente Comprimento (Atenção MIL) [controle nulo]              0.668  ép  9  (5s)
    Cobertura DELFI 5 Mb (MLP)                                     0.838  ép  6  (1s)
    Caduceus + Comprimento (Projeção → Atenção MIL)                0.739  ép  1  (4s)
    Caduceus + Comprimento (Projeção → Média → MLP)                0.799  ép 38 

In [4]:
# =============================================================================
# ===== CÉLULA 4 — ENCODER, GENERALIZAÇÃO E COORTE ============================
# =============================================================================

print("=" * 96)
print("CÉLULA 4 — ENCODER, GENERALIZAÇÃO E COORTE")
print("=" * 96)

RES = {}
for k in CHAVES:
    m = np.array([r[k]["media"] for r in REPLICAS])
    auc_ens = float(roc_auc_score(Y_DEV, OOF_ENS[k]))
    # Concordância entre réplicas: correlação de Spearman média par a par entre
    # os escores das cinco réplicas nos mesmos pacientes. Mede se elas ORDENAM
    # os pacientes da mesma forma.
    oofs = [r[k]["oof"] for r in REPLICAS]
    pares = [spearmanr(oofs[i], oofs[j]).correlation
             for i in range(len(oofs)) for j in range(i+1, len(oofs))]
    RES[k] = dict(media=float(m.mean()), dp_sementes=float(m.std(ddof=1)),
                  oof=OOF_ENS[k], auc_oof_ens=auc_ens,
                  ganho_ensemble=auc_ens - float(m.mean()),
                  concordancia=float(np.mean(pares)),
                  externo=np.mean([r[k]["externo"] for r in REPLICAS], axis=0),
                  externo_por_semente=[r[k]["externo"] for r in REPLICAS])

tab_int = pd.DataFrame([dict(Braco=nome(k), AUROC=RES[k]["media"],
                             DP_entre_sementes=RES[k]["dp_sementes"],
                             AUROC_ensemble=RES[k]["auc_oof_ens"],
                             Ganho_do_ensemble=RES[k]["ganho_ensemble"],
                             Concordancia_entre_replicas=RES[k]["concordancia"])
                        for k in CHAVES]).sort_values("AUROC", ascending=False)
print("\n" + "-" * 96)
print("DESEMPENHO INTERNO (BALTIMORE) — 5 SEMENTES × 5 DOBRAS")
print("'AUROC' é a média das 25 dobras. 'AUROC_ensemble' usa o escore médio das cinco")
print("réplicas, é o modelo que de fato avaliamos fora. 'Ganho_do_ensemble' mede o")
print("efeito de AGREGAR escores; 'Concordancia' mede se as réplicas ORDENAM os")
print("pacientes da mesma forma. São coisas diferentes.")
print("-" * 96)
print(tab_int.to_string(index=False, formatters={
    "AUROC": lambda v: f"{v:.3f}", "DP_entre_sementes": lambda v: f"{v:.3f}",
    "AUROC_ensemble": lambda v: f"{v:.3f}", "Ganho_do_ensemble": lambda v: f"{v:+.3f}",
    "Concordancia_entre_replicas": lambda v: f"{v:.3f}"}))

_at = [k for k in CHAVES if AGREGACAO.get(k) == "atencao"]
_md = [k for k in CHAVES if AGREGACAO.get(k) == "media_apos_projecao"]
if _at and _md:
    c_at = float(np.mean([RES[k]["concordancia"] for k in _at]))
    c_md = float(np.mean([RES[k]["concordancia"] for k in _md]))
    g_at = float(np.mean([RES[k]["ganho_ensemble"] for k in _at]))
    g_md = float(np.mean([RES[k]["ganho_ensemble"] for k in _md]))
    print(f"\n  {'':28s} {'concordância':>13s} {'ganho ensemble':>16s}")
    print(f"  {'braços com ATENÇÃO':<28s} {c_at:>13.3f} {g_at:>+16.3f}")
    print(f"  {'braços com MÉDIA':<28s} {c_md:>13.3f} {g_md:>+16.3f}")
    if c_at < c_md:
        print(f"  As réplicas dos braços de atenção ordenam os pacientes de forma")
        print(f"  menos parecida entre si do que as dos braços de média.")

# --- O QUE O ENCODER ACRESCENTA ---------------------------------------------
print("\n" + "=" * 96)
print("RESULTADO PRINCIPAL — O QUE O ENCODER ACRESCENTA")
print("=" * 96)
print(f"  Reconstrução do vetor do Caduceus:")
print(f"    só composição de 4-mers   : R² = {R2_KMER:.3f}")
print(f"    só ocupação de promotores : R² = {R2_OCUP:.3f}")
print(f"    as duas juntas            : R² = {R2_RESID:.3f}")
if RESID_OK:
    ra = RES["rep_cad_resid"]["media"]; ca = RES["cad_lin"]["media"]
    rk = RES["rep_kmer"]["media"]
    n_exc = int(np.sum(NULOS_RESID >= ra)); n_perm = len(NULOS_RESID)
    P_RESID = float((1 + n_exc) / (1 + n_perm))
    p_lo = float(beta.ppf(0.025, n_exc, n_perm - n_exc + 1)) if n_exc > 0 else 0.0
    p_hi = float(beta.ppf(0.975, n_exc + 1, n_perm - n_exc)) if n_exc < n_perm else 1.0
    gan, gan_lo, gan_hi = bootstrap_delta_auc(
        Y_DEV, OOF_ENS["rep_kmer"], OOF_ENS["cad_lin"], CFG["N_BOOTSTRAP"], SEMENTE_MESTRA)
    print(f"\n  Poder discriminante (AUROC no desenvolvimento):")
    print(f"    Vetor completo do Caduceus   : {ca:.3f}")
    print(f"    Composição de 4-mers sozinha : {rk:.3f}")
    print(f"    Apenas o resíduo             : {ra:.3f}")
    print(f"    Nulo do resíduo, percentil 95: {PERC95_RESID:.3f}")
    print(f"    p empírico : {P_RESID:.4f}   ({n_exc} de {n_perm} permutações ≥ observado)")
    print(f"    IC 95% de Monte Carlo do p : [{p_lo:.4f}, {p_hi:.4f}]")
    print(f"\n  Ganho do encoder sobre contar 4-mers : {gan:+.3f} "
          f"[IC 95% {gan_lo:+.3f}, {gan_hi:+.3f}]")
    if P_RESID <= 0.05:
        print(f"\n  LEITURA: há sinal residual sob esta reconstrução linear. Quase toda")
        print(f"  a variância do encoder é composição de bases, mas a fração que não")
        print(f"  é carrega sinal diagnóstico.")
    else:
        print(f"\n  LEITURA: sob esta reconstrução o resíduo não se distingue do nulo.")
        print(f"  O encoder funciona como recodificação da composição e da localização.")
else:
    P_RESID = p_lo = p_hi = gan = gan_lo = gan_hi = float("nan")

# --- GENERALIZAÇÃO PARA A COORTE NÃO HOMÓLOGA -------------------------------
cond_ext = META["condicao"].values[IDX_EXT]
mask_hcc = (cond_ext == HCC)
sel = mask_hcc | np.isin(cond_ext, list(SAUDAVEL))
y_ext = mask_hcc[sel].astype(int)

def ic(y, p, n_boot):
    """Bootstrap dos pacientes, com os modelos fixos."""
    rng = np.random.RandomState(SEMENTE_MESTRA); v = []
    for _ in range(n_boot):
        i = rng.randint(0, len(y), len(y))
        if len(np.unique(y[i])) > 1:
            v.append(roc_auc_score(y[i], p[i]))
    return (float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))) if v else (np.nan, np.nan)

print("\n" + "#" * 96)
print("#  GENERALIZAÇÃO PARA UMA COORTE NÃO HOMÓLOGA")
print("#  Carcinoma hepatocelular contra saudáveis, num tipo tumoral ausente do")
print("#  treino e numa população com outra ascendência, centro e protocolo.")
print("#" * 96)

linhas = []
for k in CHAVES:
    por_sem = np.array([roc_auc_score(y_ext, e[sel]) for e in RES[k]["externo_por_semente"]])
    a = roc_auc_score(y_ext, RES[k]["externo"][sel])
    lo, hi = ic(y_ext, RES[k]["externo"][sel], CFG["N_BOOTSTRAP"])
    RES[k]["auc_ext"] = a
    linhas.append(dict(Braco=nome(k), AUROC_externa=a, DP_sementes=float(por_sem.std(ddof=1)),
                       IC_inf=lo, IC_sup=hi, AUROC_interna=RES[k]["media"],
                       Variacao=a - RES[k]["media"]))
tab_ext = pd.DataFrame(linhas).sort_values("AUROC_externa", ascending=False)
print(f"\n{int(y_ext.sum())} casos / {int((1-y_ext).sum())} controles")
print("-" * 96)
print(tab_ext.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

print("\n" + "-" * 96)
print(f"PONTO DE OPERAÇÃO EXPLORATÓRIO — limiar fixado em "
      f"{100*CFG['ESPECIFICIDADE_ALVO']:.0f}% de especificidade no desenvolvimento")
print("-" * 96)
linhas_op = []
for k in CHAVES:
    lim = np.percentile(RES[k]["oof"][Y_DEV == 0], 100 * CFG["ESPECIFICIDADE_ALVO"])
    p = RES[k]["externo"][sel]
    linhas_op.append(dict(Braco=nome(k),
        Especificidade=float(((p < lim) & (y_ext == 0)).sum() / max((y_ext == 0).sum(), 1)),
        Sensibilidade=float(((p >= lim) & (y_ext == 1)).sum() / max((y_ext == 1).sum(), 1))))
tab_op = pd.DataFrame(linhas_op).sort_values("Sensibilidade", ascending=False)
print(tab_op.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("  A especificidade mostrada é a obtida na coorte externa, não a pedida.")

# --- O QUE A ATTENTION/MIL FAZ -----------------------------------------------------
print("\n" + "-" * 96)
print("O QUE A ATTENTION/MIL FAZ")
print("A justificativa usual do Attention-MIL é que ele encontra as instâncias")
print("informativas. 'N efetivo' = 1 / soma dos pesos ao quadrado: a quantos")
print(f"fragmentos de peso igual a distribuição equivale. Uniforme seria {CFG['N_FRAGMENTOS']}.")
print("-" * 96)
linhas_at = []
for b in NEURAIS:
    todos = [w for r in REPLICAS for w in r[b]["pesos"]]
    if not todos: continue
    n_ef = [float(np.median(1.0 / (w ** 2).sum(axis=1))) for w in todos]
    rho  = [float(np.nanmedian([spearmanr(w[i], COMP_ORD[N_DEV:][i]).correlation
                                for i in range(min(40, len(w)))])) for w in todos]
    linhas_at.append(dict(Braco=nome(b), N_efetivo=float(np.median(n_ef)),
                          Fracao_da_bolsa=float(np.median(n_ef))/CFG["N_FRAGMENTOS"],
                          Correlacao_com_comprimento=float(np.median(rho))))
tab_at = pd.DataFrame(linhas_at)
print(tab_at.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

# --- AS COORTES SÃO DISTINGUÍVEIS ENTRE PESSOAS SAUDÁVEIS? -------------------
print("\n" + "=" * 96)
print("QUANTO DE CADA REPRESENTAÇÃO É ASSINATURA DE COORTE?")
print("Só indivíduos SAUDÁVEIS das duas coortes. O rótulo é a coorte, não a doença.")
print("Com quantas direções principais se separa um saudável de Baltimore de um de")
print("Hong Kong? Menos componentes = diferença de coorte mais dominante.")
print("=" * 96)
sd_ = np.where((META["estudo"].values == ESTUDO_DEV) & META["condicao"].isin(SAUDAVEL).values)[0]
se_ = np.where((META["estudo"].values == ESTUDO_EXT) & META["condicao"].isin(SAUDAVEL).values)[0]
loc = POS[np.concatenate([sd_, se_])]
y_co = np.r_[np.zeros(len(sd_)), np.ones(len(se_))].astype(int)
print(f"  {len(sd_)} saudáveis de Baltimore contra {len(se_)} de Hong Kong "
      f"(classes desbalanceadas: a regressão usa pesos compensados)")

KL = sorted(CFG["K_COMPONENTES_PCA"]); KMAX = max(KL)
skfc = StratifiedKFold(n_splits=CFG["N_DOBRAS"], shuffle=True, random_state=SEMENTE_MESTRA)
linhas_co = []
for nm, Xr in [("Cobertura DELFI 5 Mb", V_DELFI[loc]), ("Ocupação de Promotores", V_OCUP[loc]),
               ("Composição de 4-mers", V_KMER[loc]), ("Embedding do Caduceus", V_CAD[loc])]:
    pr = {k: np.zeros(len(y_co)) for k in KL}
    for tr, te in skfc.split(Xr, y_co):
        e = StandardScaler().fit(Xr[tr])
        Xt, Xv = e.transform(Xr[tr]), e.transform(Xr[te])
        kd = min(KMAX, len(tr)-1, Xr.shape[1])
        # Uma única decomposição, fatiada: os subespaços ficam aninhados.
        pc = PCA(n_components=kd, svd_solver="randomized",
                 random_state=SEMENTE_MESTRA).fit(Xt)
        Zt, Zv = pc.transform(Xt), pc.transform(Xv)
        for k in KL:
            kk = min(k, kd)
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", ConvergenceWarning)
                lr = LogisticRegression(max_iter=3000, class_weight="balanced").fit(
                    Zt[:, :kk], y_co[tr])
            pr[k][te] = lr.predict_proba(Zv[:, :kk])[:, 1]
    linha = dict(Representacao=nm, Dimensoes=Xr.shape[1]); k95 = None
    for k in KL:
        a = roc_auc_score(y_co, pr[k]); linha[f"k={k}"] = a
        if k95 is None and a >= 0.95: k95 = k
    linha["k_para_0.95"] = k95 if k95 else f">{KMAX}"
    linhas_co.append(linha)
tab_co = pd.DataFrame(linhas_co)
print()
print(tab_co.to_string(index=False, float_format=lambda v: f"{v:.3f}"))

# -----------------------------------------------------------------------------
print("\n" + "-" * 96)
print("LIMITES DO QUE ESTA CÉLULA MEDE")
print("-" * 96)
print("  • A queda na coorte externa confunde TRÊS causas: tipo tumoral, ascendência")
print("    e protocolo, que este conjunto de dados não separa. Não é validação de")
print("    uma mesma tarefa clínica, nem validação clínica de espécie alguma.")
print(f"  • Os máximos de AUROC foram escolhidos depois de ver os dados. Com "
      f"{len(CHAVES)} braços,")
print("    o vencedor carrega viés de seleção; as sementes descrevem a instabilidade,")
print("    não a removem.")
print("  • 'N efetivo' é propriedade da DISTRIBUIÇÃO DE PESOS, não uma contagem de")
print("    fragmentos informativos. São conceitos diferentes.")
print("  • O teste de coorte NÃO estabelece causa: as coortes diferem em ascendência,")
print("    centro, protocolo e provavelmente idade, sem metadados para separá-los.")
print("  • O ganho do encoder sobre 4-mers é um contraste entre representações, não")
print("    a identificação do pré-treino como causa desse ganho.")

print("\n" + "=" * 96); print("FIM DA CÉLULA 4"); print("=" * 96)

# =============================================================================
# ===== FIM CÉLULA 4 ==========================================================
# =============================================================================

CÉLULA 4 — ENCODER, GENERALIZAÇÃO E COORTE

------------------------------------------------------------------------------------------------
DESEMPENHO INTERNO (BALTIMORE) — 5 SEMENTES × 5 DOBRAS
'AUROC' é a média das 25 dobras. 'AUROC_ensemble' usa o escore médio das cinco
réplicas, é o modelo que de fato avaliamos fora. 'Ganho_do_ensemble' mede o
efeito de AGREGAR escores; 'Concordancia' mede se as réplicas ORDENAM os
pacientes da mesma forma. São coisas diferentes.
------------------------------------------------------------------------------------------------
                                                          Braco AUROC DP_entre_sementes AUROC_ensemble Ganho_do_ensemble Concordancia_entre_replicas
                Caduceus + Comprimento (Média → Projeção → MLP) 0.857             0.009          0.865            +0.008                       0.925
                           Caduceus + Comprimento (Média → MLP) 0.854             0.008          0.857            +0.003            

In [5]:
# =============================================================================
# ===== CÉLULA 5 — FIGURAS, EXPORTAÇÃO E SÍNTESE ==============================
# =============================================================================

print("=" * 96); print("CÉLULA 5 — FIGURAS, EXPORTAÇÃO E SÍNTESE"); print("=" * 96)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 9,
                     "axes.titlesize": 10, "axes.titleweight": "bold",
                     "axes.spines.top": False, "axes.spines.right": False,
                     "savefig.bbox": "tight", "font.family": "DejaVu Sans"})
C_CAD, C_KMER, C_DELFI = "#2166AC", "#B2182B", "#1B7837"
C_SOBE, C_DESCE = "#4393C3", "#D6604D"
inv = {nome(k): k for k in CHAVES}

# ============================ FIGURA 1 =======================================
fig = plt.figure(figsize=(15.0, 6.6))
gs = fig.add_gridspec(1, 2, width_ratios=[2.1, 1], wspace=0.28)
ax = fig.add_subplot(gs[0, 0])

base = RES["cad_lin"]["media"]
acum, pos, alt, bas, cor, eti, erro = base, [], [], [], [], [], []
for i, d in enumerate(degraus):
    pos.append(i); alt.append(d["delta"]); erro.append(d["dp_sementes"])
    bas.append(acum if d["delta"] > 0 else acum + d["delta"])
    cor.append(C_SOBE if d["delta"] > 0 else C_DESCE)
    eti.append(d["etiqueta"]); acum += d["delta"]

ax.bar([-1], [base], color="#999999", width=0.62)
ax.bar(pos, np.abs(alt), bottom=bas, color=cor, width=0.62,
       yerr=erro, capsize=3, error_kw=dict(lw=1.0, ecolor="#333333"))
ax.bar([len(degraus)], [acum], color="#333333", width=0.62)
topos = [b + abs(h) + e for b, h, e in zip(bas, alt, erro)] + [base, acum]
pisos = list(bas) + [base, acum]
folga = max(0.04, 0.22 * (max(topos) - min(pisos)))
ax.set_ylim(min(pisos) - folga * 1.15, max(topos) + folga)
for p_, h, b_, e in zip(pos, alt, bas, erro):
    ax.text(p_, b_ + abs(h) + e + 0.006, f"{h:+.3f}", ha="center", va="bottom",
            fontsize=8.5, fontweight="bold")
ax.text(-1, base + 0.006, f"{base:.3f}", ha="center", va="bottom",
        fontsize=8.5, fontweight="bold")
ax.text(len(degraus), acum + 0.006, f"{acum:.3f}", ha="center", va="bottom",
        fontsize=8.5, fontweight="bold")
ax.set_xticks([-1] + pos + [len(degraus)])
ax.set_xticklabels([CURTO["cad_lin"]] + eti + [CURTO["cad_att"]], rotation=34,
                   ha="right", fontsize=8)
ax.set_ylabel("AUROC no desenvolvimento interno")
ax.set_title("A · A escada: cada barra muda uma única coisa", loc="left")
i_p = int(np.argmin(alt))
ax.annotate(f"maior perda\n({eti[i_p]})", xy=(i_p, bas[i_p]),
            xytext=(i_p - 0.60, min(pisos) - folga * 0.85), fontsize=8.5,
            fontweight="bold", color=C_KMER,
            arrowprops=dict(arrowstyle="->", color=C_KMER, lw=1.2))
ax.set_xlabel(f"Soma dos degraus {soma:+.4f} = diferença entre extremos {extremos:+.4f} "
              f"(identidade aritmética)\nbarras = média das diferenças por dobra · "
              f"erro = desvio entre {len(CFG['SEMENTES'])} sementes",
              fontsize=8, style="italic", labelpad=14)

ax = fig.add_subplot(gs[0, 1])
w, xs = 0.35, np.arange(len(replicacao))
ax.bar(xs - w/2, [r["caduceus"] for r in replicacao], w,
       yerr=[r["dp_cad"] for r in replicacao], label="Caduceus", color=C_CAD, capsize=3)
ax.bar(xs + w/2, [r["kmers"] for r in replicacao], w,
       yerr=[r["dp_kmer"] for r in replicacao], label="4-mers", color=C_KMER, capsize=3)
vs = [r["caduceus"] for r in replicacao] + [r["kmers"] for r in replicacao]
mg = 0.45 * (max(vs) - min(vs) + 1e-6)
ax.set_ylim(min(vs) - mg, max(vs) + mg); ax.axhline(0, c="black", lw=0.9)
ax.set_xticks(xs); ax.set_xticklabels([r["degrau"] for r in replicacao], fontsize=9)
ax.set_ylabel("Δ AUROC")
ax.set_title("B · Os mesmos degraus, noutra representação", loc="left")
ax.legend(fontsize=8, frameon=False)

fig.text(0.01, -0.15,
         "Cinza: modelo mais simples (média dos embeddings + regressão logística). Preto: modelo completo "
         "(Attention-MIL). Barras azuis sobem, vermelhas descem.\n"
         "NORMALIZAÇÃO troca a referência de padronização; CABEÇA troca a regressão por uma rede; PROJEÇÃO acrescenta uma camada "
         "DEPOIS da média; ORDEM move essa mesma camada para ANTES;\n"
         "AGREGADOR troca a média por atenção. Como a projeção é afim, ela comuta com a média para quaisquer pesos: as duas "
         "arquiteturas do degrau ORDEM diferem só na posição da ReLU.\n"
         "A decomposição depende do caminho: outra ordem dos degraus daria outras fatias com a mesma soma. O teste por bootstrap "
         "pareado está na tabela da Célula 3.",
         fontsize=7.4, style="italic", va="top")
fig.suptitle("Figura 1 — Onde o sinal se perde na cadeia de agregação", fontsize=12,
             fontweight="bold", y=1.02)
fig.savefig(SAIDA / "Figura_1_Escada.png"); fig.savefig(SAIDA / "Figura_1_Escada.pdf")
plt.close(fig); print("  [FIGURA] Figura_1_Escada")

# ============================ FIGURA 2 =======================================
def espalha(v, minimo):
    v = np.asarray(v, float); o = np.argsort(v); s = v.copy()
    for k in range(1, len(o)):
        if s[o[k]] - s[o[k-1]] < minimo: s[o[k]] = s[o[k-1]] + minimo
    return s - (s.mean() - v.mean())

fig, ax = plt.subplots(figsize=(11.0, 8.0))
ks = [inv[n] for n in tab_int["Braco"].tolist()]
yi = np.array([RES[k]["media"] for k in ks]); ye = np.array([RES[k]["auc_ext"] for k in ks])
faixa = max(yi.max(), ye.max()) - min(yi.min(), ye.min())
re_, rd_ = espalha(yi, 0.032*faixa), espalha(ye, 0.032*faixa)
for i, k in enumerate(ks):
    n = nome(k)
    if "DELFI" in n:        c, lw, al = C_DELFI, 2.6, 1.0
    elif "Caduceus" in n:   c, lw, al = C_CAD, 1.5, 0.85
    elif "4-mers" in n:     c, lw, al = C_KMER, 1.5, 0.85
    else:                   c, lw, al = "#777777", 1.1, 0.6
    ax.plot([0, 1], [yi[i], ye[i]], "-o", color=c, lw=lw, ms=5, alpha=al)
    ax.plot([-0.045, -0.012], [re_[i], yi[i]], "-", color=c, lw=0.5, alpha=0.5)
    ax.text(-0.050, re_[i], CURTO[k], ha="right", va="center", fontsize=7.4, color=c)
    ax.plot([1.012, 1.045], [ye[i], rd_[i]], "-", color=c, lw=0.5, alpha=0.5)
    ax.text(1.050, rd_[i], f"{ye[i]:.3f}", ha="left", va="center", fontsize=7.4, color=c)
todos = np.r_[yi, ye, re_, rd_, 0.5]; mg = 0.05*(todos.max()-todos.min())
ax.set_ylim(todos.min()-mg, todos.max()+mg)
ax.axhline(0.5, ls="--", c="black", lw=1.0)
ax.text(0.5, 0.5+0.10*mg, "acaso", ha="center", fontsize=8, style="italic")
ax.set_xlim(-0.70, 1.34); ax.set_xticks([0, 1])
ax.set_xticklabels([f"Baltimore (n={N_DEV})\npan-câncer vs saudável",
                    f"Hong Kong (n={int(sel.sum())})\ncarcinoma hepatocelular vs saudável"],
                   fontsize=9)
ax.set_ylabel("AUROC")
ax.set_title("Figura 2 — O que sobrevive à mudança de coorte E de tarefa", fontweight="bold")
fig.text(0.01, -0.03,
         "Cada linha liga o desempenho de um modelo nas duas coortes. Verde: cobertura em megabases. "
         "Azul: representações do Caduceus. Vermelho: 4-mers. Cinza: controles.\n"
         "ATENÇÃO: isto NÃO é validação externa da mesma tarefa. O alvo externo é um tipo tumoral ausente do treino, numa "
         "população com outra ascendência, centro e protocolo. A queda confunde as três causas.",
         fontsize=7.6, style="italic", va="top")
fig.savefig(SAIDA / "Figura_2_Generalizacao.png"); fig.savefig(SAIDA / "Figura_2_Generalizacao.pdf")
plt.close(fig); print("  [FIGURA] Figura_2_Generalizacao")

# ============================ FIGURA 3 =======================================
b_i, b_e = inv[tab_int.iloc[0]["Braco"]], inv[tab_ext.iloc[0]["Braco"]]
if b_i == b_e: b_e = inv[tab_ext.iloc[1]["Braco"]]
papel = {b_i: "maior AUROC no desenvolvimento", b_e: "maior AUROC na coorte externa"}
gr = ["healthy", "hepatitis_b", "cirrhosis", HCC]
bc = {"healthy": "Saudável", "hepatitis_b": "Hepatite B", "cirrhosis": "Cirrose",
      HCC: "Carcinoma\nhepatocelular"}
nc = {g: f"{bc[g]}\n(n={int((cond_ext == g).sum())})" for g in gr}
mk = np.isin(cond_ext, gr)
fig, axes = plt.subplots(1, 2, figsize=(13.0, 6.0))
for ax, b in zip(np.atleast_1d(axes), [b_i, b_e]):
    dd = pd.DataFrame(dict(grupo=[nc[c] for c in cond_ext[mk]], escore=RES[b]["externo"][mk]))
    sns.violinplot(data=dd, x="grupo", y="escore", order=[nc[g] for g in gr], ax=ax,
                   inner="quartile", cut=0, hue="grupo", legend=False,
                   palette=["#BBBBBB", "#F4A582", "#D6604D", "#B2182B"])
    sns.stripplot(data=dd, x="grupo", y="escore", order=[nc[g] for g in gr], ax=ax,
                  color="black", size=2.0, alpha=0.35, jitter=0.22)
    ax.set_title(f"{nome(b)}\n({papel[b]})\nAUROC externa {RES[b]['auc_ext']:.3f}", fontsize=8.5)
    ax.set_xlabel(""); ax.set_ylabel("Probabilidade predita de câncer")
    ax.tick_params(axis="x", labelsize=8.5)
fig.suptitle("Figura 3 — Como os modelos se comportam diante de doença hepática sem tumor",
             fontsize=12, fontweight="bold")
fig.text(0.01, -0.05,
         "Cada ponto é um paciente; as linhas dentro do violino são os quartis. Hepatite B e cirrose são doenças "
         "do fígado SEM tumor e não entram no cálculo da AUROC:\nsua posição mostra se o modelo confunde inflamação com câncer. "
         "Os dois painéis mostram as maiores AUROC observadas, escolhidas depois de ver os resultados.",
         fontsize=7.6, style="italic", va="top")
fig.tight_layout()
fig.savefig(SAIDA / "Figura_3_Grupos_Clinicos.png"); fig.savefig(SAIDA / "Figura_3_Grupos_Clinicos.pdf")
plt.close(fig); print("  [FIGURA] Figura_3_Grupos_Clinicos")

# ============================ FIGURA 4 =======================================
# A dimensão exibida é escolhida em 40 pacientes sorteados e mostrada em outros
# 40, também sorteados, nunca nas primeiras linhas da tabela, que poderiam
# refletir ordem de lote ou de centro em vez de acaso.
rng_f4 = np.random.default_rng(SEMENTE_MESTRA + 999)
pt, nt = IDX_DEV[Y_DEV == 1], IDX_DEV[Y_DEV == 0]
perm_p, perm_n = rng_f4.permutation(pt), rng_f4.permutation(nt)
esc_p, esc_n = np.sort(perm_p[:20]), np.sort(perm_n[:20])
exi_p, exi_n = np.sort(perm_p[20:40]), np.sort(perm_n[20:40])
if len(exi_p) < 5 or len(exi_n) < 5:
    exi_p, exi_n = esc_p, esc_n
dim = int(np.argmax(np.abs(np.asarray(CAD[esc_p], np.float32).mean(axis=(0, 1))
                           - np.asarray(CAD[esc_n], np.float32).mean(axis=(0, 1)))))
Bp = np.asarray(CAD[exi_p], np.float32); Bn = np.asarray(CAD[exi_n], np.float32)
J = np.concatenate([Bp, Bn]); mu, sd = J.mean(axis=(0, 1)), J.std(axis=(0, 1)) + 1e-9
del J
Bp, Bn = (Bp - mu)/sd, (Bn - mu)/sd

fp, fn_ = Bp[:, :, dim].ravel(), Bn[:, :, dim].ravel()
mp, mn  = Bp[:, :, dim].mean(axis=1), Bn[:, :, dim].mean(axis=1)
d_frag_fig = float(np.abs(fp.mean() - fn_.mean()) /
                   (np.sqrt((fp.var(ddof=1) + fn_.var(ddof=1))/2) + 1e-12))
d_pac_fig  = float(np.abs(mp.mean() - mn.mean()) /
                   (np.sqrt((mp.var(ddof=1) + mn.var(ddof=1))/2) + 1e-12))

fig, axes = plt.subplots(1, 3, figsize=(16.0, 4.9))
ax = axes[0]
sns.kdeplot(fn_, ax=ax, fill=True, color=C_DESCE, label="controle", lw=1.4)
sns.kdeplot(fp,  ax=ax, fill=True, color=C_SOBE,  label="caso", lw=1.4)
ax.set_title(f"A · Um fragmento isolado\nd = {d_frag_fig:.4f} (dimensão {dim})", loc="left")
ax.set_xlabel(f"dimensão {dim}, padronizada"); ax.set_ylabel("densidade")
ax.legend(fontsize=8, frameon=False)

ax = axes[1]
sns.kdeplot(mn, ax=ax, fill=True, color=C_DESCE, label="controle", lw=1.4)
sns.kdeplot(mp, ax=ax, fill=True, color=C_SOBE,  label="caso", lw=1.4)
ax.set_title(f"B · Média de {CFG['N_FRAGMENTOS']} fragmentos\n"
             f"d = {d_pac_fig:.3f} (dimensão {dim})", loc="left")
ax.set_xlabel("média por paciente"); ax.set_ylabel("")
ax.legend(fontsize=8, frameon=False)

ax = axes[2]
T = np.concatenate([Bn[:, :, dim], Bp[:, :, dim]])
antes, depois = np.maximum(T, 0).mean(axis=1), np.maximum(T.mean(axis=1), 0)
ax.scatter(depois, antes, s=30, c=C_CAD, alpha=0.7, edgecolors="white", lw=0.4, zorder=3)
lim = [min(depois.min(), antes.min())-0.05, max(depois.max(), antes.max())+0.05]
ax.plot(lim, lim, "--", c="black", lw=1.2, zorder=4); ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_title("C · A ordem importa", loc="left")
ax.set_xlabel("ReLU DEPOIS da média"); ax.set_ylabel("ReLU ANTES da média")
ax.text(0.04, 0.95, f"dispersão entre pacientes\nReLU antes:  {np.std(antes):.4f}\n"
        f"ReLU depois: {np.std(depois):.4f}", transform=ax.transAxes, fontsize=8,
        va="top", bbox=dict(boxstyle="round,pad=0.4", fc="#F7F7F7", ec="#CCCCCC"))
fig.suptitle("Figura 4 — Por que a ordem das operações importa", fontsize=12, fontweight="bold")
fig.text(0.01, -0.07,
         f"COMO LER — A e B mostram UMA dimensão, escolhida em 40 pacientes sorteados e exibida em outros 40; os d são os desta "
         f"dimensão, não a mediana das {D_EMB}.\nSobre o desenvolvimento (Célula 1), o erro de estimar a média domina até cerca de "
         f"{N_ESTRELA:.0f} fragmentos de {CFG['N_FRAGMENTOS']} [IC 95% {N_EST_LO:.0f}–{N_EST_HI:.0f}]: os fragmentos de um mesmo "
         f"paciente são correlacionados.\nC: cada ponto é um paciente; se aplicar a ReLU antes ou depois da média fosse indiferente, "
         f"todos cairiam na diagonal. A camada linear comuta com a média,\nentão o desvio da diagonal vem inteiramente da ReLU. "
         f"O custo diagnóstico dessa escolha é o que o degrau ORDEM da Figura 1 mede.",
         fontsize=7.4, style="italic", va="top")
fig.tight_layout()
fig.savefig(SAIDA / "Figura_4_Mecanismo.png"); fig.savefig(SAIDA / "Figura_4_Mecanismo.pdf")
plt.close(fig); print("  [FIGURA] Figura_4_Mecanismo")
del Bp, Bn

# ---------------------------- exportação -------------------------------------
tab_int.to_csv(SAIDA/"desempenho_interno.csv", index=False)
tab_ext.to_csv(SAIDA/"desempenho_externo.csv", index=False)
tab_op.to_csv(SAIDA/"ponto_de_operacao.csv", index=False)
tab_co.to_csv(SAIDA/"artefato_de_coorte.csv", index=False)
tab_at.to_csv(SAIDA/"diagnostico_atencao.csv", index=False)
tab_clip.to_csv(SAIDA/"controles_otimizacao.csv", index=False)
pd.DataFrame(degraus).to_csv(SAIDA/"escada.csv", index=False)
pd.DataFrame(replicacao).to_csv(SAIDA/"replicacao_degraus.csv", index=False)
pd.DataFrame({nome(k): [r[k]["media"] for r in REPLICAS] for k in CHAVES},
             index=[f"semente_{s}" for s in CFG["SEMENTES"]]).to_csv(
                 SAIDA/"auroc_por_semente.csv")


HASH_CODIGO = "ver notebook depositado"
for cand in ["__notebook__.ipynb", "__notebook_source__.ipynb"]:
    p_nb = Path("/kaggle/working") / cand
    if p_nb.exists():
        HASH_CODIGO = hashlib.sha256(p_nb.read_bytes()).hexdigest()[:16]; break

def _num(x):
    """JSON estrito não aceita NaN; devolvemos None quando o valor não existe."""
    if x is None: return None
    v = float(x)
    return None if not np.isfinite(v) else v

json.dump(dict(
    assinatura_configuracao=HASH_CFG, assinatura_codigo=HASH_CODIGO,
    assinatura_dados=HASH_DADOS, arquivos=hashes,
    configuracao={k: str(v) for k, v in CFG.items()}, mapa_condicao=MAPA,
    tarefa_interna="pan-cancer vs saudavel (Baltimore)",
    tarefa_externa="carcinoma hepatocelular vs saudavel (Hong Kong)",
    tarefas_identicas=bool(hcc_dev > 0),
    coorte_externa="EXPLORATORIA_NAO_HOMOLOGA",
    forense_restrita_ao_desenvolvimento=True,
    n_pacientes=int(N_PAC), n_desenvolvimento=int(N_DEV), n_externo=int(len(IDX_EXT)),
    n_bracos=len(CHAVES),
    gate_padding=int(n_zero), gate_vetor_identico={k: _num(v) for k, v in ERROS.items()},
    gate_comutacao_linear_media=_num(ERRO_COMUTA),
    gate_canal_nulo=_num(dp_nulo), gate_telescopio=_num(discrep),
    gate_finitude="completa: DELFI, comprimentos; triagem: Caduceus, 4-mers",
    fracao_fora_histograma=_num(frac_fora), esparsidade_ocupacao=_num(frac_zeros),
    crossover_variancia=_num(N_ESTRELA),
    crossover_variancia_ic95=[_num(N_EST_LO), _num(N_EST_HI)],
    crossover_por_representacao={k: _num(v["n_estrela"]) for k, v in forense.items()},
    d_cohen_n_por_classe=int(n_d),
    d_cohen_fragmento=_num(D_FRAG), d_cohen_paciente=_num(D_PAC),
    r2_reconstrucao=_num(R2_RESID), r2_so_kmers=_num(R2_KMER),
    r2_so_ocupacao=_num(R2_OCUP), r2_so_ocupacao_padronizada=_num(R2_OCUP_PAD),
    residuo_computavel=bool(RESID_OK), residuo_interpretavel=bool(RESID_INTERPRE),
    nulo_percentil95_residuo=_num(PERC95_RESID),
    nulo_percentil95_caduceus=_num(PERC95_CAD),
    p_empirico_residuo=_num(P_RESID),
    p_empirico_ic_monte_carlo=[_num(p_lo), _num(p_hi)],
    ganho_encoder_sobre_kmers=_num(gan),
    ganho_encoder_ic95=[_num(gan_lo), _num(gan_hi)],
    limiar_parada=CFG["LIMIAR_PARADA"],
    inflacao_pico_bruto_sobre_suave=_num(INFLACAO_PICO),
    execucoes_abaixo_do_limiar=N_EXEC_ABAIXO,
    comparacoes_afetadas_pelo_limiar=N_COMP_AFETADAS,
    taxa_maxima_recorte=_num(TAXA_CLIP), avisos_convergencia=int(AVISOS_CONV),
    degraus={d["etiqueta"]: dict(
        delta_por_dobra=_num(d["delta"]), dp_sementes=_num(d["dp_sementes"]),
        delta_escore_agrupado=_num(d["boot_delta"]),
        ic95_bootstrap=[_num(d["boot_inf"]), _num(d["boot_sup"])],
        negativos=int(d["neg"]), de=int(d["n_comp"]),
        delta_atingiram=_num(d["delta_ok"]), n_atingiram=int(d["n_ok"]))
        for d in degraus},
    replicacao={r["degrau"]: dict(caduceus=_num(r["caduceus"]), kmers=_num(r["kmers"]),
                kmers_ic95=[_num(r["boot_kmer_inf"]), _num(r["boot_kmer_sup"])])
                for r in replicacao},
    concordancia_entre_replicas={nome(k): _num(RES[k]["concordancia"]) for k in CHAVES},
    dimensao_figura4=int(dim),
    verificacoes_extra=bool(CFG["VERIFICACOES_EXTRA"]),
    tempo_total_segundos=round(time.time()-T_INICIO, 1)),
    open(SAIDA/"procedencia.json", "w"), indent=2, ensure_ascii=False)

# ---------------------------- síntese ----------------------------------------
print("\n" + "=" * 96); print("SÍNTESE"); print("=" * 96)

print(f"\n1. O ENCODER. O vetor do Caduceus é reconstruível a partir da composição de")
print(f"   4-mers com R² = {R2_KMER:.3f}; somar a localização leva a {R2_RESID:.3f}.")
if RESID_OK:
    print(f"   O resíduo discrimina a {RES['rep_cad_resid']['media']:.3f} contra um nulo "
          f"empírico construído")
    print(f"   sobre o próprio resíduo, com o mesmo desenho de réplicas")
    print(f"   (p = {P_RESID:.4f}, IC de Monte Carlo [{p_lo:.4f}, {p_hi:.4f}]).")
    print(f"   Ganho sobre contar 4-mers: {gan:+.3f} de AUROC "
          f"[IC 95% {gan_lo:+.3f}, {gan_hi:+.3f}].")
    print(f"   Formulação: HÁ SINAL RESIDUAL SOB ESTA RECONSTRUÇÃO LINEAR.")

print(f"\n2. A BOLSA. Um fragmento isolado separa os grupos por d = {D_FRAG:.4f}; a média")
print(f"   por paciente, por d = {D_PAC:.4f} (medianas das {D_EMB} dimensões, "
      f"{n_d} casos e {n_d} controles).")
print(f"   O erro de estimar a média domina até cerca de {N_ESTRELA:.0f} fragmentos de "
      f"{CFG['N_FRAGMENTOS']}")
print(f"   [IC 95% {N_EST_LO:.0f}–{N_EST_HI:.0f}], sob correlação intra-bolsa homogênea.")

print(f"\n3. A ESCADA. Maior perda em {pior['etiqueta']}.")
print(f"   Média das diferenças por dobra : {pior['delta']:+.3f} ± {pior['dp_sementes']:.3f} "
      f"(pacientes fixos)")
print(f"   Diferença do escore agrupado   : {pior['boot_delta']:+.3f} "
      f"[IC 95% {pior['boot_inf']:+.3f}, {pior['boot_sup']:+.3f}] (modelos fixos)")
print(f"   Negativo em {pior['neg']} de {pior['n_comp']} comparações.")
if np.isfinite(pior["delta_ok"]) and pior["n_ok"] < pior["n_comp"]:
    print(f"   Restrito às {pior['n_ok']} execuções acima do limiar: {pior['delta_ok']:+.3f} "
          f"({pior['neg_ok']}/{pior['n_ok']} neg).")
print(f"   A projeção comuta com a média para quaisquer pesos (erro {ERRO_COMUTA:.1e}):")
print(f"   as duas arquiteturas diferem só na posição da ReLU.")

n_item = 4
for r in replicacao:
    print(f"\n{n_item}. Degrau {r['degrau']} em 4-mers: {r['kmers']:+.3f} ± {r['dp_kmer']:.3f} "
          f"[IC 95% {r['boot_kmer_inf']:+.3f}, {r['boot_kmer_sup']:+.3f}],")
    print(f"   {r['neg_kmer']}/{r['n_kmer']} negativos, "
          f"{'mesmo sinal' if r['mesmo_sinal'] else 'sinal oposto'} ao Caduceus.")
    n_item += 1

print(f"\n{n_item}. GENERALIZAÇÃO. Maior AUROC interna: {tab_int.iloc[0]['Braco']} "
      f"({tab_int.iloc[0]['AUROC']:.3f}).")
print(f"   Maior AUROC externa: {tab_ext.iloc[0]['Braco']} "
      f"({tab_ext.iloc[0]['AUROC_externa']:.3f}).")
print(f"   O primeiro atinge "
      f"{float(tab_ext.set_index('Braco')['AUROC_externa'].loc[tab_int.iloc[0]['Braco']]):.3f} "
      f"fora. A coorte externa não é homóloga;")
print(f"   ambos os máximos foram escolhidos depois de ver os dados."); n_item += 1

print(f"\n{n_item}. PONTO DE OPERAÇÃO EXPLORATÓRIO. Maior sensibilidade observada:")
print(f"   {tab_op.iloc[0]['Braco']}, especificidade "
      f"{tab_op.iloc[0]['Especificidade']:.3f} e sensibilidade "
      f"{tab_op.iloc[0]['Sensibilidade']:.3f}")
print(f"   em {int(y_ext.sum())} casos e {int((1-y_ext).sum())} controles. "
      f"Não constitui validação clínica.")

print("\n" + "-" * 96); print("ARQUIVOS GERADOS"); print("-" * 96)
for a in sorted(SAIDA.iterdir()):
    print(f"  {a.name:<40s} {a.stat().st_size/1024:8.1f} KB")
print(f"\nTempo total: {(time.time()-T_INICIO)/60:.1f} minutos")
print("=" * 96); print("FIM"); print("=" * 96)

# =============================================================================
# ===== FIM CÉLULA 5 ==========================================================
# =============================================================================

CÉLULA 5 — FIGURAS, EXPORTAÇÃO E SÍNTESE
  [FIGURA] Figura_1_Escada
  [FIGURA] Figura_2_Generalizacao
  [FIGURA] Figura_3_Grupos_Clinicos
  [FIGURA] Figura_4_Mecanismo

SÍNTESE

1. O ENCODER. O vetor do Caduceus é reconstruível a partir da composição de
   4-mers com R² = 0.990; somar a localização leva a 0.990.
   O resíduo discrimina a 0.604 contra um nulo empírico construído
   sobre o próprio resíduo, com o mesmo desenho de réplicas
   (p = 0.0050, IC de Monte Carlo [0.0000, 0.0183]).
   Ganho sobre contar 4-mers: +0.098 de AUROC [IC 95% +0.054, +0.143].
   Formulação: HÁ SINAL RESIDUAL SOB ESTA RECONSTRUÇÃO LINEAR.

2. A BOLSA. Um fragmento isolado separa os grupos por d = 0.0104; a média
   por paciente, por d = 0.2031 (medianas das 256 dimensões, 60 casos e 60 controles).
   O erro de estimar a média domina até cerca de 675 fragmentos de 5000
   [IC 95% 509–1012], sob correlação intra-bolsa homogênea.

3. A ESCADA. Maior perda em ORDEM.
   Média das diferenças por dobra : -0.156